# S&P 500 Değerleme ve Makine Öğrenmesi — Rev 2

Bu notebook, güncel S&P 500 şirketlerini aşağıdaki üç bakış açısıyla karşılaştırır:

1. **Temel değerleme:** sektör medyan F/K, FD/FAVÖK ve serbest nakit akışı verimi.
2. **Analist görüşü:** güncel hedef fiyatlar ve tavsiye dağılımı.
3. **Makine öğrenmesi:** geçmiş fiyat örüntülerinden S&P 500'e göre göreceli sıralama.

Sonuç, hisseleri kesin getiri vaadiyle değil, araştırma önceliği veren göreceli bir **0–100 puan** ile sıralar. Modelin test sonuçları sınırlıdır; çıktı yatırım tavsiyesi değildir.


## 1. Kurulum

Gerekli Python paketlerini kurar ve notebook boyunca kullanılacak kütüphaneleri yükler. Yeni bir Colab oturumunda ilk olarak bu hücre çalıştırılmalıdır.


In [1]:
!pip install yfinance pandas tqdm openpyxl pyarrow scikit-learn scipy -q

from io import StringIO
from datetime import datetime
import time
import requests
import numpy as np
import pandas as pd
import yfinance as yf
from tqdm.auto import tqdm


## 2. Güncel S&P 500 şirket evreni

Wikipedia'daki güncel S&P 500 listesini indirir; sembol, şirket adı, sektör ve alt sektör kolonlarını standartlaştırır. Nokta içeren semboller Yahoo Finance biçimine dönüştürülür.


In [2]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

sp500_raw = pd.read_html(StringIO(response.text))[0]

sp500_universe = sp500_raw.rename(columns={
    "Symbol": "ticker",
    "Security": "company_name",
    "GICS Sector": "sector_sp500",
    "GICS Sub-Industry": "industry_sp500"
})

sp500_universe["yf_ticker"] = sp500_universe["ticker"].str.replace(
    ".", "-", regex=False
)

sp500_universe = sp500_universe[[
    "ticker", "yf_ticker", "company_name", "sector_sp500", "industry_sp500"
]]

print("Şirket sayısı:", len(sp500_universe))
display(sp500_universe.head())


Şirket sayısı: 503


,ticker,yf_ticker,company_name,sector_sp500,industry_sp500
0,MMM,MMM,3M,Industrials,Industrial Conglomerates
1,AOS,AOS,A. O. Smith,Industrials,Building Products
2,ABT,ABT,Abbott Laboratories,Health Care,Health Care Equipment
3,ABBV,ABBV,AbbVie,Health Care,Biotechnology
4,ACN,ACN,Accenture,Information Technology,IT Consulting & Other Services


## 3. Güncel piyasa snapshot fonksiyonları

Her şirketin güncel fiyatını, piyasa değerini ve işlem hacmini Yahoo Finance üzerinden alacak yardımcı fonksiyonları tanımlar. Bu hücre yalnızca fonksiyonları oluşturur; henüz veri çekmez.


In [3]:
#Yardımcı fonksiyonlar
def human_number(value):
    if value is None:
        return None

    try:
        value = float(value)

        if value >= 1_000_000_000_000:
            return f"{value / 1_000_000_000_000:.2f}T"
        elif value >= 1_000_000_000:
            return f"{value / 1_000_000_000:.2f}B"
        elif value >= 1_000_000:
            return f"{value / 1_000_000:.2f}M"
        elif value >= 1_000:
            return f"{value / 1_000:.2f}K"

        return f"{value:.2f}"

    except:
        return None
        #Şirket profil bilgileri
def get_company_profile(info):
    return {
        "currency": info.get("currency"),
        "quote_type": info.get("quoteType"),
        "exchange": info.get("exchange"),
        "sector": info.get("sector"),
        "industry": info.get("industry"),
        "long_name": info.get("longName")
    }
    #Piyasa snapshot bilgileri
def get_market_values(info):
    price = (
        info.get("currentPrice")
        or info.get("regularMarketPrice")
        or info.get("previousClose")
    )

    avg_volume = (
        info.get("averageVolume")
        or info.get("averageVolume10days")
        or info.get("averageDailyVolume10Day")
    )

    market_cap = info.get("marketCap")

    if price is not None:
        price = round(price, 2)

    if avg_volume is not None:
        avg_volume = round(avg_volume)

    if market_cap is not None:
        market_cap = round(market_cap)

    avg_daily_dollar_volume = None

    if price is not None and avg_volume is not None:
        avg_daily_dollar_volume = round(price * avg_volume)

    return {
        "market_cap": market_cap,
        "current_price": price,
        "average_volume": avg_volume,
        "avg_daily_dollar_volume": avg_daily_dollar_volume,

        "market_cap_display": human_number(market_cap),
        "average_volume_display": human_number(avg_volume),
        "avg_daily_dollar_volume_display": human_number(avg_daily_dollar_volume)
    }
    #Ana snapshot fonksiyonu

def get_company_snapshot(yf_ticker):
    try:
        t = yf.Ticker(yf_ticker)
        info = t.get_info()

        profile = get_company_profile(info)
        market_values = get_market_values(info)

        return {
            "yf_ticker": yf_ticker,
            **market_values,
            **profile,
            "error": None
        }

    except Exception as e:
        return {
            "yf_ticker": yf_ticker,

            "market_cap": None,
            "current_price": None,
            "average_volume": None,
            "avg_daily_dollar_volume": None,

            "market_cap_display": None,
            "average_volume_display": None,
            "avg_daily_dollar_volume_display": None,

            "currency": None,
            "quote_type": None,
            "exchange": None,
            "sector": None,
            "industry": None,
            "long_name": None,

            "error": str(e)
        }


## 4. Güncel piyasa verilerini çekme

S&P 500 şirketlerinin güncel piyasa verilerini toplar, şirket evreniyle birleştirir ve modelde kullanılmayan tekrarlı profil alanlarını kaldırır. İşlem birkaç dakika sürebilir.


In [4]:
results = []

tickers = sp500_universe["yf_ticker"].tolist()

for yf_ticker in tqdm(tickers):
    snapshot = get_company_snapshot(yf_ticker)
    results.append(snapshot)
    time.sleep(0.1)

sp500_snapshot_yf = pd.DataFrame(results)

sp500_snapshot = sp500_universe.merge(
    sp500_snapshot_yf,
    on="yf_ticker",
    how="left"
)

sp500_snapshot.head()

sp500_snapshot["error"].value_counts(dropna=False)

drop_columns = [
    "quote_type",
    "sector",
    "industry",
    "long_name"
]

sp500_snapshot = sp500_snapshot.drop(
    columns=drop_columns,
    errors="ignore"
)

print(sp500_snapshot.shape)
display(sp500_snapshot.head())


  0%|          | 0/503 [00:00<?, ?it/s]

(503, 15)


,ticker,yf_ticker,company_name,sector_sp500,industry_sp500,market_cap,current_price,average_volume,avg_daily_dollar_volume,market_cap_display,average_volume_display,avg_daily_dollar_volume_display,currency,exchange,error
0,MMM,MMM,3M,Industrials,Industrial Conglomerates,8.693017e+10,168.56,3442880,580331853,86.93B,3.44M,580.33M,USD,NYQ,None
1,AOS,AOS,A. O. Smith,Industrials,Building Products,8.219750e+09,60.48,1670515,101032747,8.22B,1.67M,101.03M,USD,NYQ,None
2,ABT,ABT,Abbott Laboratories,Health Care,Health Care Equipment,1.874524e+11,108.33,11230274,1216575582,187.45B,11.23M,1.22B,USD,NYQ,None
3,ABBV,ABBV,AbbVie,Health Care,Biotechnology,4.531949e+11,256.46,6380811,1636422789,453.19B,6.38M,1.64B,USD,NYQ,None
4,ACN,ACN,Accenture,Information Technology,IT Consulting & Other Services,1.142618e+11,186.72,8374146,1563620541,114.26B,8.37M,1.56B,USD,NYQ,None


## 5. Finansal göstergeler ve analist verileri

Hasılat, FAVÖK, nakit akışı, büyüme, kârlılık, borçluluk, değerleme oranları ve güncel analist hedeflerini çıkaran fonksiyonu tanımlar. Oranlar mümkün olduğunca sayısal tutulur; eksik kaynak verileri `NaN` bırakılır.


In [5]:
import pandas as pd
import numpy as np
import yfinance as yf
import time

from tqdm.auto import tqdm


def to_pct(value):
    """0.15 değerini %15'e çevirir."""
    if value is None or pd.isna(value):
        return np.nan

    return round(float(value) * 100, 2)


def safe_divide(numerator, denominator, multiplier=1):
    if (
        numerator is None
        or denominator is None
        or pd.isna(numerator)
        or pd.isna(denominator)
        or denominator == 0
    ):
        return np.nan

    return round(
        float(numerator) / float(denominator) * multiplier,
        2
    )


def get_fundamental_metrics(yf_ticker):
    try:
        ticker_object = yf.Ticker(yf_ticker)
        info = ticker_object.get_info()

        market_cap = info.get("marketCap")
        current_price = (
            info.get("currentPrice")
            or info.get("regularMarketPrice")
            or info.get("previousClose")
        )

        revenue = info.get("totalRevenue")
        ebitda = info.get("ebitda")
        operating_cash_flow = info.get("operatingCashflow")
        free_cash_flow = info.get("freeCashflow")

        total_cash = info.get("totalCash")
        total_debt = info.get("totalDebt")

        net_debt = np.nan

        if total_debt is not None and total_cash is not None:
            net_debt = total_debt - total_cash

        # EBITDA pozitif değilse borç/FAVÖK hesaplanmaz
        net_debt_ebitda = np.nan

        if (
            pd.notna(net_debt)
            and ebitda is not None
            and ebitda > 0
        ):
            net_debt_ebitda = round(net_debt / ebitda, 2)

        trailing_pe = info.get("trailingPE")

        earnings_yield_pct = np.nan

        if trailing_pe is not None and trailing_pe > 0:
            earnings_yield_pct = round(100 / trailing_pe, 2)

        fcf_yield_pct = safe_divide(
            free_cash_flow,
            market_cap,
            multiplier=100
        )

        # Güncel analist hedefleri
        analyst_targets = {}

        try:
            targets_result = ticker_object.analyst_price_targets

            if isinstance(targets_result, dict):
                analyst_targets = targets_result

        except Exception:
            analyst_targets = {}

        target_mean = analyst_targets.get("mean")
        target_median = analyst_targets.get("median")
        target_low = analyst_targets.get("low")
        target_high = analyst_targets.get("high")

        analyst_upside_pct = np.nan

        if (
            target_mean is not None
            and current_price is not None
            and current_price > 0
        ):
            analyst_upside_pct = round(
                (target_mean / current_price - 1) * 100,
                2
            )

        analyst_dispersion_pct = np.nan

        if (
            target_high is not None
            and target_low is not None
            and target_mean is not None
            and target_mean > 0
        ):
            analyst_dispersion_pct = round(
                (target_high - target_low)
                / target_mean
                * 100,
                2
            )

        # Buy / Hold / Sell sayıları
        strong_buy = np.nan
        buy = np.nan
        hold = np.nan
        sell = np.nan
        strong_sell = np.nan
        analyst_count = np.nan

        try:
            recommendations = (
                ticker_object.recommendations_summary
            )

            if (
                recommendations is not None
                and not recommendations.empty
            ):
                current_recommendation = recommendations.iloc[0]

                strong_buy = current_recommendation.get(
                    "strongBuy",
                    np.nan
                )
                buy = current_recommendation.get("buy", np.nan)
                hold = current_recommendation.get("hold", np.nan)
                sell = current_recommendation.get("sell", np.nan)
                strong_sell = current_recommendation.get(
                    "strongSell",
                    np.nan
                )

                recommendation_counts = [
                    strong_buy,
                    buy,
                    hold,
                    sell,
                    strong_sell
                ]

                analyst_count = sum(
                    value
                    for value in recommendation_counts
                    if pd.notna(value)
                )

        except Exception:
            pass

        return {
            "yf_ticker": yf_ticker,

            # Büyüklük
            "revenue": revenue,
            "ebitda": ebitda,
            "operating_cash_flow": operating_cash_flow,
            "free_cash_flow": free_cash_flow,

            # Büyüme
            "revenue_growth_pct": to_pct(
                info.get("revenueGrowth")
            ),
            "earnings_growth_pct": to_pct(
                info.get("earningsGrowth")
            ),

            # Kârlılık
            "gross_margin_pct": to_pct(
                info.get("grossMargins")
            ),
            "operating_margin_pct": to_pct(
                info.get("operatingMargins")
            ),
            "ebitda_margin_pct": to_pct(
                info.get("ebitdaMargins")
            ),
            "net_margin_pct": to_pct(
                info.get("profitMargins")
            ),
            "roe_pct": to_pct(
                info.get("returnOnEquity")
            ),
            "roa_pct": to_pct(
                info.get("returnOnAssets")
            ),

            # Borç
            "total_cash": total_cash,
            "total_debt": total_debt,
            "net_debt": net_debt,
            "net_debt_ebitda": net_debt_ebitda,
            "debt_to_equity": info.get("debtToEquity"),
            "current_ratio": info.get("currentRatio"),

            # Değerleme
            "trailing_eps": info.get("trailingEps"),
            "forward_eps": info.get("forwardEps"),
            "trailing_pe": trailing_pe,
            "forward_pe": info.get("forwardPE"),
            "peg_ratio": (
                info.get("trailingPegRatio")
                or info.get("pegRatio")
            ),
            "ev_ebitda": info.get("enterpriseToEbitda"),
            "ev_revenue": info.get("enterpriseToRevenue"),
            "price_to_book": info.get("priceToBook"),
            "fcf_yield_pct": fcf_yield_pct,
            "earnings_yield_pct": earnings_yield_pct,
            "beta": info.get("beta"),

            # Analist
            "analyst_target_mean": target_mean,
            "analyst_target_median": target_median,
            "analyst_target_low": target_low,
            "analyst_target_high": target_high,
            "analyst_upside_pct": analyst_upside_pct,
            "analyst_dispersion_pct": analyst_dispersion_pct,
            "analyst_count": analyst_count,
            "strong_buy_count": strong_buy,
            "buy_count": buy,
            "hold_count": hold,
            "sell_count": sell,
            "strong_sell_count": strong_sell,

            "fundamental_error": None
        }

    except Exception as error:
        return {
            "yf_ticker": yf_ticker,
            "fundamental_error": str(error)
        }


## 6. Finansal verileri bütün şirketler için çekme

Fonksiyonu bütün şirketlere uygular, her 50 şirkette geçici kontrol dosyası yazar, hataları raporlar ve sonuçları `sp500_snapshot` tablosuna ekler.


In [6]:
fundamental_rows = []

all_tickers = (
    sp500_snapshot["yf_ticker"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

for index, symbol in enumerate(
    tqdm(all_tickers),
    start=1
):
    result = get_fundamental_metrics(symbol)
    fundamental_rows.append(result)

    # Yahoo'yu çok sık çağırmamak için
    time.sleep(0.20)

    # Her 50 şirkette geçici kayıt
    if index % 50 == 0:
        pd.DataFrame(fundamental_rows).to_csv(
            "/content/fundamentals_checkpoint.csv",
            index=False
        )

fundamentals_df = pd.DataFrame(
    fundamental_rows
)

print("Çekilen şirket:", len(fundamentals_df))
display(fundamentals_df.head())

failed_fundamentals = fundamentals_df[
    fundamentals_df["fundamental_error"].notna()
]

print(
    "Başarılı:",
    fundamentals_df["fundamental_error"].isna().sum()
)

print(
    "Hatalı:",
    len(failed_fundamentals)
)

display(
    failed_fundamentals[
        ["yf_ticker", "fundamental_error"]
    ]
)

new_columns = [
    column
    for column in fundamentals_df.columns
    if column != "yf_ticker"
]

sp500_snapshot = sp500_snapshot.drop(
    columns=[
        column
        for column in new_columns
        if column in sp500_snapshot.columns
    ],
    errors="ignore"
)

sp500_snapshot = sp500_snapshot.merge(
    fundamentals_df,
    on="yf_ticker",
    how="left",
    validate="one_to_one"
)

print("Yeni snapshot boyutu:", sp500_snapshot.shape)
display(sp500_snapshot.head())


  0%|          | 0/503 [00:00<?, ?it/s]

ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: FOX"}}}


Çekilen şirket: 503


,yf_ticker,revenue,ebitda,operating_cash_flow,free_cash_flow,revenue_growth_pct,earnings_growth_pct,gross_margin_pct,operating_margin_pct,ebitda_margin_pct,...,analyst_target_high,analyst_upside_pct,analyst_dispersion_pct,analyst_count,strong_buy_count,buy_count,hold_count,sell_count,strong_sell_count,fundamental_error
0,MMM,2.518000e+10,6.488000e+09,4.899000e+09,6.350125e+09,2.5,32.8,39.41,20.35,25.77,...,219.0,10.09,53.35,18.0,1.0,8.0,6.0,0.0,3.0,None
1,AOS,3.804900e+09,7.837000e+08,6.923000e+08,4.846125e+08,-0.7,-15.0,38.59,18.93,20.60,...,84.0,15.59,35.76,13.0,1.0,4.0,6.0,1.0,1.0,None
2,ABT,4.658500e+10,1.168100e+10,9.905000e+09,7.209125e+09,13.0,-47.5,56.85,14.71,25.07,...,135.0,10.96,26.62,27.0,4.0,17.0,6.0,0.0,0.0,None
3,ABBV,6.438600e+10,3.076300e+10,1.950700e+10,1.687025e+10,10.2,290.4,72.79,40.03,47.78,...,328.0,7.85,46.28,31.0,8.0,17.0,5.0,0.0,1.0,None
4,ACN,7.310059e+10,1.294377e+10,1.318209e+10,1.208938e+10,5.6,9.0,32.01,16.96,17.71,...,275.0,-1.36,78.72,27.0,3.0,11.0,13.0,0.0,0.0,None


Başarılı: 503
Hatalı: 0


,yf_ticker,fundamental_error


Yeni snapshot boyutu: (503, 57)


,ticker,yf_ticker,company_name,sector_sp500,industry_sp500,market_cap,current_price,average_volume,avg_daily_dollar_volume,market_cap_display,...,analyst_target_high,analyst_upside_pct,analyst_dispersion_pct,analyst_count,strong_buy_count,buy_count,hold_count,sell_count,strong_sell_count,fundamental_error
0,MMM,MMM,3M,Industrials,Industrial Conglomerates,8.693017e+10,168.56,3442880,580331853,86.93B,...,219.0,10.09,53.35,18.0,1.0,8.0,6.0,0.0,3.0,None
1,AOS,AOS,A. O. Smith,Industrials,Building Products,8.219750e+09,60.48,1670515,101032747,8.22B,...,84.0,15.59,35.76,13.0,1.0,4.0,6.0,1.0,1.0,None
2,ABT,ABT,Abbott Laboratories,Health Care,Health Care Equipment,1.874524e+11,108.33,11230274,1216575582,187.45B,...,135.0,10.96,26.62,27.0,4.0,17.0,6.0,0.0,0.0,None
3,ABBV,ABBV,AbbVie,Health Care,Biotechnology,4.531949e+11,256.46,6380811,1636422789,453.19B,...,328.0,7.85,46.28,31.0,8.0,17.0,5.0,0.0,1.0,None
4,ACN,ACN,Accenture,Information Technology,IT Consulting & Other Services,1.142618e+11,186.72,8374146,1563620541,114.26B,...,275.0,-1.36,78.72,27.0,3.0,11.0,13.0,0.0,0.0,None


## 7. Okunabilir görüntüleme tablosu

Büyük parasal değerleri K/M/B/T biçimine getirir. Makine öğrenmesinde kullanılacak `sp500_snapshot` değişmez; biçimlendirme yalnızca `sp500_display` kopyasına uygulanır.


In [7]:
def compact_number(value):
    if pd.isna(value):
        return ""

    value = float(value)

    if abs(value) >= 1_000_000_000_000:
        return f"{value / 1_000_000_000_000:.2f}T"

    if abs(value) >= 1_000_000_000:
        return f"{value / 1_000_000_000:.2f}B"

    if abs(value) >= 1_000_000:
        return f"{value / 1_000_000:.2f}M"

    if abs(value) >= 1_000:
        return f"{value / 1_000:.2f}K"

    return f"{value:.2f}"


# Görüntüleme tablosunu yeniden oluştur
sp500_display = sp500_snapshot.copy()

# Eski ve tekrarlı display kolonlarını kaldır
sp500_display = sp500_display.drop(
    columns=[
        "market_cap_display",
        "average_volume_display",
        "avg_daily_dollar_volume_display"
    ],
    errors="ignore"
)

# Büyük değerleri doğrudan mevcut kolonlarda formatla
large_columns = [
    "market_cap",
    "average_volume",
    "avg_daily_dollar_volume",
    "revenue",
    "ebitda",
    "operating_cash_flow",
    "free_cash_flow",
    "total_cash",
    "total_debt",
    "net_debt"
]

for column in large_columns:
    if column in sp500_display.columns:
        sp500_display[column] = (
            sp500_display[column]
            .apply(compact_number)
        )

# Diğer sayısal kolonları iki basamağa yuvarla
for column in sp500_display.select_dtypes(
    include="number"
).columns:
    sp500_display[column] = (
        sp500_display[column].round(2)
    )

display(sp500_display.head(20))


,ticker,yf_ticker,company_name,sector_sp500,industry_sp500,market_cap,current_price,average_volume,avg_daily_dollar_volume,currency,...,analyst_target_high,analyst_upside_pct,analyst_dispersion_pct,analyst_count,strong_buy_count,buy_count,hold_count,sell_count,strong_sell_count,fundamental_error
0,MMM,MMM,3M,Industrials,Industrial Conglomerates,86.93B,168.56,3.44M,580.33M,USD,...,219.0,10.09,53.35,18.0,1.0,8.0,6.0,0.0,3.0,None
1,AOS,AOS,A. O. Smith,Industrials,Building Products,8.22B,60.48,1.67M,101.03M,USD,...,84.0,15.59,35.76,13.0,1.0,4.0,6.0,1.0,1.0,None
2,ABT,ABT,Abbott Laboratories,Health Care,Health Care Equipment,187.45B,108.33,11.23M,1.22B,USD,...,135.0,10.96,26.62,27.0,4.0,17.0,6.0,0.0,0.0,None
3,ABBV,ABBV,AbbVie,Health Care,Biotechnology,453.19B,256.46,6.38M,1.64B,USD,...,328.0,7.85,46.28,31.0,8.0,17.0,5.0,0.0,1.0,None
4,ACN,ACN,Accenture,Information Technology,IT Consulting & Other Services,114.26B,186.72,8.37M,1.56B,USD,...,275.0,-1.36,78.72,27.0,3.0,11.0,13.0,0.0,0.0,None
5,ADBE,ADBE,Adobe Inc.,Information Technology,Application Software,105.94B,266.51,6.21M,1.65B,USD,...,380.0,3.78,68.70,40.0,4.0,8.0,23.0,4.0,1.0,None
6,AMD,AMD,Advanced Micro Devices,Information Technology,Semiconductors,779.62B,477.57,26.15M,12.49B,USD,...,1250.0,28.54,144.17,54.0,4.0,39.0,11.0,0.0,0.0,None
7,AES,AES,AES Corporation,Utilities,Independent Power Producers & Energy Traders,10.55B,14.79,7.81M,115.47M,USD,...,15.0,1.42,0.00,10.0,0.0,0.0,10.0,0.0,0.0,None
8,AFL,AFL,Aflac,Financials,Life & Health Insurance,58.76B,117.21,2.28M,266.90M,USD,...,138.0,1.13,32.90,15.0,2.0,1.0,8.0,3.0,1.0,None
9,A,A,Agilent Technologies,Health Care,Life Sciences Tools & Services,42.54B,150.86,2.21M,333.90M,USD,...,190.0,16.30,19.95,22.0,4.0,12.0,6.0,0.0,0.0,None


## 8. Temel değerleme hedefleri

Şirketleri kendi sektör medyanlarıyla karşılaştırır. F/K, FD/FAVÖK ve serbest nakit akışı verimi yöntemlerinden geçerli olanlarla hedef fiyat üretir ve bunların ortalamasını temel analiz hedefi olarak hesaplar.


In [8]:
import numpy as np
import pandas as pd

df = sp500_snapshot.copy()

# Hesaplamada kullanılacak kolonları sayısala çevir
numeric_columns = [
    "current_price",
    "market_cap",
    "forward_eps",
    "forward_pe",
    "ebitda",
    "ev_ebitda",
    "free_cash_flow",
    "fcf_yield_pct",
    "net_debt",
    "analyst_target_mean"
]

for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce"
        )


# Yaklaşık dolaşımdaki hisse sayısı
df["calculated_shares"] = (
    df["market_cap"] / df["current_price"]
)

df.loc[
    (df["market_cap"] <= 0)
    | (df["current_price"] <= 0),
    "calculated_shares"
] = np.nan


# Aşırı ve anlamsız oranları sektör medyanına alma
df["valid_forward_pe"] = df["forward_pe"].where(
    df["forward_pe"].between(1, 100)
)

df["valid_ev_ebitda"] = df["ev_ebitda"].where(
    df["ev_ebitda"].between(1, 50)
)

df["valid_fcf_yield"] = df["fcf_yield_pct"].where(
    df["fcf_yield_pct"].between(0.1, 30)
)


# Sektörel medyanlar
df["sector_forward_pe"] = (
    df.groupby("sector_sp500")[
        "valid_forward_pe"
    ].transform("median")
)

df["sector_ev_ebitda"] = (
    df.groupby("sector_sp500")[
        "valid_ev_ebitda"
    ].transform("median")
)

df["sector_fcf_yield_pct"] = (
    df.groupby("sector_sp500")[
        "valid_fcf_yield"
    ].transform("median")
)


# 1. F/K yöntemine göre hedef fiyat
df["target_price_pe"] = (
    df["forward_eps"]
    * df["sector_forward_pe"]
)

df.loc[
    (df["forward_eps"] <= 0)
    | (df["target_price_pe"] <= 0),
    "target_price_pe"
] = np.nan


# 2. FD/FAVÖK yöntemine göre hedef fiyat
estimated_enterprise_value = (
    df["ebitda"]
    * df["sector_ev_ebitda"]
)

estimated_equity_value = (
    estimated_enterprise_value
    - df["net_debt"]
)

df["target_price_ev_ebitda"] = (
    estimated_equity_value
    / df["calculated_shares"]
)

df.loc[
    (df["ebitda"] <= 0)
    | (df["calculated_shares"] <= 0)
    | (df["target_price_ev_ebitda"] <= 0),
    "target_price_ev_ebitda"
] = np.nan


# 3. Serbest nakit akışı yöntemine göre hedef
estimated_market_cap_fcf = (
    df["free_cash_flow"]
    / (df["sector_fcf_yield_pct"] / 100)
)

df["target_price_fcf"] = (
    estimated_market_cap_fcf
    / df["calculated_shares"]
)

df.loc[
    (df["free_cash_flow"] <= 0)
    | (df["calculated_shares"] <= 0)
    | (df["target_price_fcf"] <= 0),
    "target_price_fcf"
] = np.nan


# Geçerli değerleme yöntemlerinin ortalaması
valuation_columns = [
    "target_price_pe",
    "target_price_ev_ebitda",
    "target_price_fcf"
]

df["valuation_method_count"] = (
    df[valuation_columns].notna().sum(axis=1)
)

df["fundamental_target_mean"] = (
    df[valuation_columns].mean(
        axis=1,
        skipna=True
    )
)


# Bizim hesapladığımız yükseliş beklentisi
df["fundamental_upside_pct"] = (
    (
        df["fundamental_target_mean"]
        / df["current_price"]
    ) - 1
) * 100


# Analist hedefi ile bizim hedefimizin farkı
df["target_difference_pct"] = (
    (
        df["fundamental_target_mean"]
        - df["analyst_target_mean"]
    ).abs()
    / df["analyst_target_mean"]
) * 100


# Analist ile temel değerleme tutarlılığı
df["analyst_fundamental_consistency_pct"] = (
    100 - df["target_difference_pct"]
).clip(lower=0, upper=100)


# Yönler aynı mı?
df["fundamental_direction"] = np.where(
    df["fundamental_upside_pct"] >= 0,
    "UP",
    "DOWN"
)

df["analyst_direction"] = np.where(
    df["analyst_upside_pct"] >= 0,
    "UP",
    "DOWN"
)

df["direction_agreement"] = (
    df["fundamental_direction"]
    == df["analyst_direction"]
)


# Sonuçları yuvarla
result_columns = [
    "sector_forward_pe",
    "sector_ev_ebitda",
    "sector_fcf_yield_pct",
    "target_price_pe",
    "target_price_ev_ebitda",
    "target_price_fcf",
    "fundamental_target_mean",
    "fundamental_upside_pct",
    "target_difference_pct",
    "analyst_fundamental_consistency_pct"
]

df[result_columns] = df[result_columns].round(2)


# Ana snapshot tablosunu güncelle
sp500_snapshot = df.copy()


## 9. Temel değerleme kontrol tablosu

Üç değerleme yönteminin ürettiği hedefleri yan yana gösterir. `valuation_method_count`, bir şirket için kaç yöntemin kullanılabildiğini belirtir.


In [9]:
valuation_result = sp500_snapshot[[
    "ticker",
    "company_name",
    "sector_sp500",
    "current_price",

    "target_price_pe",
    "target_price_ev_ebitda",
    "target_price_fcf",
    "valuation_method_count",

    "fundamental_target_mean",
    "fundamental_upside_pct",

    "analyst_target_mean",
    "analyst_upside_pct",

    "analyst_fundamental_consistency_pct",
    "direction_agreement"
]].copy()

display(
    valuation_result.sort_values(
        "analyst_fundamental_consistency_pct",
        ascending=False
    ).head(30)
)


,ticker,company_name,sector_sp500,current_price,target_price_pe,target_price_ev_ebitda,target_price_fcf,valuation_method_count,fundamental_target_mean,fundamental_upside_pct,analyst_target_mean,analyst_upside_pct,analyst_fundamental_consistency_pct,direction_agreement
50,ADSK,Autodesk,Information Technology,217.90,257.05,217.90,470.64,3,315.20,44.65,315.36765,44.73,99.95,True
162,ECHO,EchoStar,Communication Services,89.80,226.70,29.68,NaN,2,128.19,42.75,128.42857,43.02,99.81,True
214,GD,General Dynamics,Industrials,359.39,373.40,388.46,494.32,3,418.73,16.51,420.02475,16.87,99.69,True
268,KVUE,Kenvue,Consumer Staples,18.74,19.21,19.17,20.31,3,19.56,4.38,19.50000,4.06,99.68,True
428,TMUS,T-Mobile US,Communication Services,181.52,209.33,304.39,212.85,3,242.19,33.42,243.37500,34.08,99.51,True
75,BF.B,Brown–Forman,Consumer Staples,26.73,26.50,28.24,30.56,3,28.43,6.37,28.25882,5.72,99.39,True
349,ON,ON Semiconductor,Information Technology,74.38,81.90,103.72,132.56,3,106.06,42.59,107.08000,43.96,99.05,True
390,REGN,Regeneron Pharmaceuticals,Health Care,827.72,1030.89,736.61,786.61,3,851.37,2.86,840.42850,1.54,98.70,True
389,REG,Regency Centers,Real Estate,75.26,86.17,81.65,97.10,3,88.31,17.33,87.05556,15.67,98.56,True
168,EMR,Emerson Electric,Industrials,152.82,145.67,159.68,201.96,3,169.10,10.65,171.81444,12.43,98.42,True


## 10. Temel değerleme–analist karşılaştırması

Bizim temel değerleme hedefimizle analist ortalama hedefini simetrik yüzde farkla karşılaştırır. Ayrıca iki yaklaşımın yükseliş, düşüş veya nötr yönde uzlaşıp uzlaşmadığını hesaplar.


In [10]:
comparison_df = sp500_snapshot.copy()

required_columns = [
    "current_price",
    "fundamental_target_mean",
    "fundamental_upside_pct",
    "analyst_target_mean",
    "analyst_upside_pct"
]

for column in required_columns:
    comparison_df[column] = pd.to_numeric(
        comparison_df[column],
        errors="coerce"
    )


# Simetrik hedef fiyat farkı
average_target = (
    comparison_df["fundamental_target_mean"]
    + comparison_df["analyst_target_mean"]
) / 2

comparison_df["target_gap_pct"] = (
    (
        comparison_df["fundamental_target_mean"]
        - comparison_df["analyst_target_mean"]
    ).abs()
    / average_target
    * 100
)


# 0–100 arasında tutarlılık puanı
comparison_df["consistency_score"] = (
    100 - comparison_df["target_gap_pct"]
).clip(lower=0, upper=100)


# Bizim ve analistlerin tahmin yönü
comparison_df["fundamental_direction"] = np.select(
    [
        comparison_df["fundamental_upside_pct"] > 2,
        comparison_df["fundamental_upside_pct"] < -2
    ],
    [
        "UP",
        "DOWN"
    ],
    default="NEUTRAL"
)

comparison_df["analyst_direction"] = np.select(
    [
        comparison_df["analyst_upside_pct"] > 2,
        comparison_df["analyst_upside_pct"] < -2
    ],
    [
        "UP",
        "DOWN"
    ],
    default="NEUTRAL"
)


comparison_df["direction_agreement"] = (
    comparison_df["fundamental_direction"]
    == comparison_df["analyst_direction"]
)


# Tutarlılık seviyesi
comparison_df["consistency_level"] = pd.cut(
    comparison_df["consistency_score"],
    bins=[-1, 60, 80, 90, 100],
    labels=[
        "LOW",
        "MEDIUM",
        "HIGH",
        "VERY HIGH"
    ]
)


# Temel değerleme ile analist hedefinin ortalaması
comparison_df["combined_target_price"] = (
    comparison_df[
        [
            "fundamental_target_mean",
            "analyst_target_mean"
        ]
    ].mean(axis=1)
)


comparison_df["combined_upside_pct"] = (
    (
        comparison_df["combined_target_price"]
        / comparison_df["current_price"]
    ) - 1
) * 100


# Sayıları yuvarla
round_columns = [
    "fundamental_target_mean",
    "fundamental_upside_pct",
    "analyst_target_mean",
    "analyst_upside_pct",
    "target_gap_pct",
    "consistency_score",
    "combined_target_price",
    "combined_upside_pct"
]

comparison_df[round_columns] = (
    comparison_df[round_columns].round(2)
)

comparison_columns = [
    "yf_ticker",
    "target_gap_pct",
    "consistency_score",
    "consistency_level",
    "fundamental_direction",
    "analyst_direction",
    "direction_agreement",
    "combined_target_price",
    "combined_upside_pct"
]

sp500_snapshot = sp500_snapshot.drop(
    columns=[
        column
        for column in comparison_columns
        if column != "yf_ticker"
        and column in sp500_snapshot.columns
    ],
    errors="ignore"
)

sp500_snapshot = sp500_snapshot.merge(
    comparison_df[comparison_columns],
    on="yf_ticker",
    how="left",
    validate="one_to_one"
)


## 11. Analist karşılaştırma sonuçları

Hedef fiyat, potansiyel, analist sayısı, hedef dağılımı ve tutarlılık skorlarını birlikte gösterir. Bu tablo güncel bir karşılaştırmadır; tarihsel analist doğruluk testi değildir.


In [11]:
analyst_comparison = sp500_snapshot[[
    "ticker",
    "company_name",
    "sector_sp500",
    "current_price",

    "fundamental_target_mean",
    "fundamental_upside_pct",

    "analyst_target_mean",
    "analyst_upside_pct",
    "analyst_count",
    "analyst_dispersion_pct",

    "target_gap_pct",
    "consistency_score",
    "consistency_level",
    "direction_agreement",

    "combined_target_price",
    "combined_upside_pct",
    "valuation_method_count"
]].copy()

analyst_comparison = analyst_comparison.dropna(
    subset=[
        "fundamental_target_mean",
        "analyst_target_mean"
    ]
)

display(
    analyst_comparison.sort_values(
        [
            "direction_agreement",
            "consistency_score",
            "combined_upside_pct"
        ],
        ascending=[
            False,
            False,
            False
        ]
    ).head(50)
)


,ticker,company_name,sector_sp500,current_price,fundamental_target_mean,fundamental_upside_pct,analyst_target_mean,analyst_upside_pct,analyst_count,analyst_dispersion_pct,target_gap_pct,consistency_score,consistency_level,direction_agreement,combined_target_price,combined_upside_pct,valuation_method_count
50,ADSK,Autodesk,Information Technology,217.90,315.20,44.65,315.36765,44.73,36.0,74.67,0.05,99.95,VERY HIGH,True,315.28,44.69,3
162,ECHO,EchoStar,Communication Services,89.80,128.19,42.75,128.42857,43.02,6.0,41.27,0.19,99.81,VERY HIGH,True,128.31,42.88,2
214,GD,General Dynamics,Industrials,359.39,418.73,16.51,420.02475,16.87,24.0,34.76,0.31,99.69,VERY HIGH,True,419.38,16.69,3
268,KVUE,Kenvue,Consumer Staples,18.74,19.56,4.38,19.50000,4.06,14.0,25.64,0.31,99.69,VERY HIGH,True,19.53,4.22,3
428,TMUS,T-Mobile US,Communication Services,181.52,242.19,33.42,243.37500,34.08,27.0,53.83,0.49,99.51,VERY HIGH,True,242.78,33.75,3
75,BF.B,Brown–Forman,Consumer Staples,26.73,28.43,6.37,28.25882,5.72,18.0,53.08,0.60,99.40,VERY HIGH,True,28.34,6.04,3
349,ON,ON Semiconductor,Information Technology,74.38,106.06,42.59,107.08000,43.96,29.0,60.70,0.96,99.04,VERY HIGH,True,106.57,43.28,3
389,REG,Regency Centers,Real Estate,75.26,88.31,17.33,87.05556,15.67,20.0,22.97,1.43,98.57,VERY HIGH,True,87.68,16.51,3
168,EMR,Emerson Electric,Industrials,152.82,169.10,10.65,171.81444,12.43,29.0,58.78,1.59,98.41,VERY HIGH,True,170.46,11.54,3
120,COP,ConocoPhillips,Energy,134.26,142.99,6.50,145.33333,8.25,26.0,43.35,1.63,98.37,VERY HIGH,True,144.16,7.37,3


## 12. Güncel fiyat geçmişi

Son iki yıllık günlük fiyatları ve S&P 500 vekili olarak SPY verisini indirir. Günlük tablo geçicidir; snapshot'a yalnızca özet fiyat göstergeleri eklenir.


In [12]:
all_tickers = (
    sp500_snapshot["yf_ticker"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

download_tickers = all_tickers + ["SPY"]

recent_prices = yf.download(
    tickers=download_tickers,
    period="2y",
    interval="1d",
    auto_adjust=True,
    group_by="ticker",
    threads=True,
    progress=True
)

print("Fiyat verisi boyutu:", recent_prices.shape)


[*********************100%***********************]  504 of 504 completed


Fiyat verisi boyutu: (502, 2520)


## 13. Fiyat davranışı fonksiyonları

3/6/12 aylık getiri, oynaklık, maksimum düşüş, hareketli ortalamaya uzaklık ve hacim değişimini hesaplayan yardımcı fonksiyonları tanımlar.


In [13]:
def calculate_return(close, trading_days):
    if len(close) <= trading_days:
        return np.nan

    return round(
        (close.iloc[-1] / close.iloc[-trading_days - 1] - 1)
        * 100,
        2
    )


def calculate_price_features(
    ticker,
    price_data,
    spy_return_12m
):
    result = {
        "yf_ticker": ticker,
        "return_3m_pct": np.nan,
        "return_6m_pct": np.nan,
        "return_12m_pct": np.nan,
        "relative_return_sp500_pct": np.nan,
        "volatility_12m_pct": np.nan,
        "max_drawdown_12m_pct": np.nan,
        "price_vs_sma50_pct": np.nan,
        "price_vs_sma200_pct": np.nan,
        "volume_change_3m_pct": np.nan,
        "price_feature_error": None
    }

    try:
        ticker_data = price_data[ticker].copy()

        close = ticker_data["Close"].dropna()

        if close.empty:
            result["price_feature_error"] = (
                "Fiyat verisi bulunamadı"
            )
            return result

        result["return_3m_pct"] = calculate_return(
            close,
            63
        )

        result["return_6m_pct"] = calculate_return(
            close,
            126
        )

        result["return_12m_pct"] = calculate_return(
            close,
            252
        )

        if (
            pd.notna(result["return_12m_pct"])
            and pd.notna(spy_return_12m)
        ):
            result["relative_return_sp500_pct"] = round(
                result["return_12m_pct"]
                - spy_return_12m,
                2
            )

        # Son 12 aylık oynaklık
        daily_returns = (
            close
            .pct_change()
            .dropna()
            .tail(252)
        )

        if not daily_returns.empty:
            result["volatility_12m_pct"] = round(
                daily_returns.std()
                * np.sqrt(252)
                * 100,
                2
            )

        # Son 12 aylık maksimum düşüş
        close_12m = close.tail(252)

        if not close_12m.empty:
            running_max = close_12m.cummax()

            drawdown = (
                close_12m / running_max - 1
            )

            result["max_drawdown_12m_pct"] = round(
                drawdown.min() * 100,
                2
            )

        # Hareketli ortalamalar
        if len(close) >= 50:
            sma50 = close.tail(50).mean()

            result["price_vs_sma50_pct"] = round(
                (close.iloc[-1] / sma50 - 1) * 100,
                2
            )

        if len(close) >= 200:
            sma200 = close.tail(200).mean()

            result["price_vs_sma200_pct"] = round(
                (close.iloc[-1] / sma200 - 1) * 100,
                2
            )

        # Son üç ayın ortalama hacmini
        # önceki üç ayla karşılaştır
        if "Volume" in ticker_data.columns:
            volume = ticker_data["Volume"].dropna()

            if len(volume) >= 126:
                recent_volume = volume.tail(63).mean()

                previous_volume = (
                    volume.iloc[-126:-63].mean()
                )

                if previous_volume > 0:
                    result["volume_change_3m_pct"] = round(
                        (
                            recent_volume
                            / previous_volume
                            - 1
                        ) * 100,
                        2
                    )

        return result

    except Exception as error:
        result["price_feature_error"] = str(error)
        return result


## 14. Güncel fiyat özelliklerini snapshot'a ekleme

SPY'ın son 12 aylık getirisini hesaplar, fiyat fonksiyonlarını bütün şirketlere uygular ve sonuçları ana snapshot tablosuyla birleştirir.


In [14]:
spy_close = (
    recent_prices["SPY"]["Close"]
    .dropna()
)

spy_return_12m = calculate_return(
    spy_close,
    252
)

print(
    "SPY son 12 aylık getiri:",
    spy_return_12m,
    "%"
)

price_feature_rows = []

for ticker in tqdm(all_tickers):
    price_feature_rows.append(
        calculate_price_features(
            ticker=ticker,
            price_data=recent_prices,
            spy_return_12m=spy_return_12m
        )
    )

price_features_df = pd.DataFrame(
    price_feature_rows
)

print(
    "Hesaplanan şirket:",
    len(price_features_df)
)

display(price_features_df.head())

price_feature_columns = [
    column
    for column in price_features_df.columns
    if column != "yf_ticker"
]

sp500_snapshot = sp500_snapshot.drop(
    columns=[
        column
        for column in price_feature_columns
        if column in sp500_snapshot.columns
    ],
    errors="ignore"
)

sp500_snapshot = sp500_snapshot.merge(
    price_features_df,
    on="yf_ticker",
    how="left",
    validate="one_to_one"
)

print(
    "Güncel snapshot boyutu:",
    sp500_snapshot.shape
)


SPY son 12 aylık getiri: 19.97 %


  0%|          | 0/503 [00:00<?, ?it/s]

Hesaplanan şirket: 503


,yf_ticker,return_3m_pct,return_6m_pct,return_12m_pct,relative_return_sp500_pct,volatility_12m_pct,max_drawdown_12m_pct,price_vs_sma50_pct,price_vs_sma200_pct,volume_change_3m_pct,price_feature_error
0,MMM,10.11,10.93,10.39,-9.58,26.25,-18.77,-1.43,5.68,-10.65,None
1,AOS,6.37,-13.83,-14.88,-34.85,26.88,-30.29,-1.09,-6.32,3.90,None
2,ABT,19.80,0.21,-16.55,-36.52,26.82,-38.61,3.95,2.96,-12.09,None
3,ABBV,13.67,13.18,24.14,4.17,26.50,-17.32,0.95,13.58,-3.29,None
4,ACN,6.01,-11.36,-24.66,-44.63,44.19,-56.51,15.65,-6.62,33.78,None


Güncel snapshot boyutu: (503, 90)


## 15. Güncel birleşik görünüm

Temel değerleme, analist beklentisi ve fiyat davranışı kolonlarını birlikte göstererek ara sonuçların mantık kontrolünü sağlar.


In [15]:
final_comparison_columns = [
    "ticker",
    "company_name",
    "sector_sp500",
    "current_price",

    "fundamental_target_mean",
    "fundamental_upside_pct",

    "analyst_target_mean",
    "analyst_upside_pct",

    "consistency_score",
    "direction_agreement",

    "return_3m_pct",
    "return_6m_pct",
    "return_12m_pct",
    "relative_return_sp500_pct",
    "volatility_12m_pct",
    "max_drawdown_12m_pct",
    "price_vs_sma50_pct",
    "price_vs_sma200_pct",

    "combined_target_price",
    "combined_upside_pct"
]

final_comparison_columns = [
    column
    for column in final_comparison_columns
    if column in sp500_snapshot.columns
]

display(
    sp500_snapshot[
        final_comparison_columns
    ].sort_values(
        "combined_upside_pct",
        ascending=False
    ).head(50)
)


,ticker,company_name,sector_sp500,current_price,fundamental_target_mean,fundamental_upside_pct,analyst_target_mean,analyst_upside_pct,consistency_score,direction_agreement,return_3m_pct,return_6m_pct,return_12m_pct,relative_return_sp500_pct,volatility_12m_pct,max_drawdown_12m_pct,price_vs_sma50_pct,price_vs_sma200_pct,combined_target_price,combined_upside_pct
357,PSKY,Paramount Skydance Corporation,Communication Services,10.86,117.76,984.39,9.69286,-10.75,0.00,False,6.77,-8.52,-25.10,-45.07,53.00,-59.87,13.55,-0.73,63.73,486.80
98,CHTR,Charter Communications,Communication Services,151.99,781.84,414.40,184.41176,21.33,0.00,True,15.04,-34.55,-41.43,-61.40,50.47,-56.39,6.09,-16.57,483.13,217.87
216,GM,General Motors,Consumer Discretionary,87.76,329.45,275.40,101.64000,15.82,0.00,True,7.10,17.18,52.62,32.65,34.06,-16.00,5.55,10.38,215.54,145.61
93,CNC,Centene Corporation,Health Care,67.04,255.99,281.84,71.66667,6.90,0.00,True,7.56,53.59,134.00,114.03,48.84,-32.73,2.30,33.19,163.83,144.37
41,APTV,Aptiv,Consumer Discretionary,47.95,164.67,243.42,66.61111,38.92,15.20,True,-30.10,-34.04,-40.40,-60.37,42.92,-49.52,-10.00,-27.76,115.64,141.17
219,GPN,Global Payments,Financials,92.54,288.51,211.77,103.14286,11.46,5.34,True,40.07,21.99,7.42,-12.55,40.27,-28.99,9.33,22.80,195.83,111.61
472,VTRS,Viatris,Health Care,16.88,50.83,201.13,18.50000,9.60,6.74,True,7.09,22.02,69.20,49.23,32.30,-18.97,1.28,16.31,34.66,105.36
14,ARE,Alexandria Real Estate Equities,Real Estate,52.65,156.24,196.75,53.00000,0.66,1.32,False,4.02,6.81,-32.61,-52.58,46.71,-51.61,3.79,7.05,104.62,98.71
118,CMCSA,Comcast,Communication Services,26.49,75.11,183.55,30.08136,13.56,14.39,True,12.72,-14.88,-12.71,-32.68,30.97,-30.80,6.71,0.24,52.60,98.55
18,ALL,Allstate,Financials,259.57,745.71,187.29,274.77274,5.86,7.70,True,17.94,23.41,27.01,7.04,24.88,-11.48,1.49,18.13,510.24,96.57


## 16. Makine öğrenmesi için tarihsel fiyatlar

12 aylık geçmiş özellikler ve sonraki 12 aylık hedef oluşturabilmek için 12 yıllık günlük fiyatları indirir. Bu veri yalnızca model eğitim tablosunu üretmek için kullanılır.


In [16]:
training_tickers = (
    sp500_snapshot["yf_ticker"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

historical_prices = yf.download(
    tickers=training_tickers + ["SPY"],
    period="12y",
    interval="1d",
    auto_adjust=True,
    group_by="ticker",
    threads=True,
    progress=True
)

print(
    "Tarihsel fiyat tablosu:",
    historical_prices.shape
)


[*********************100%***********************]  504 of 504 completed


Tarihsel fiyat tablosu: (3018, 2520)


## 17. SPY aylık piyasa özellikleri

SPY günlük verisini ay sonu seviyesine indirger ve 3/6/12 aylık piyasa getirilerini hesaplar. Model böylece genel piyasa koşulunu ayrı bir özellik olarak görebilir.


In [17]:
spy_daily = historical_prices["SPY"].copy()

spy_monthly = pd.DataFrame({
    "date": spy_daily["Close"]
        .resample("ME")
        .last()
        .index,

    "spy_close": spy_daily["Close"]
        .resample("ME")
        .last()
        .values
})

spy_monthly["spy_return_3m_pct"] = (
    spy_monthly["spy_close"].pct_change(3) * 100
)

spy_monthly["spy_return_6m_pct"] = (
    spy_monthly["spy_close"].pct_change(6) * 100
)

spy_monthly["spy_return_12m_pct"] = (
    spy_monthly["spy_close"].pct_change(12) * 100
)

display(spy_monthly.tail())


,date,spy_close,spy_return_3m_pct,spy_return_6m_pct,spy_return_12m_pct
140,2026-05-31,754.536133,10.576826,11.325506,29.820984
141,2026-06-30,746.770020,15.123453,10.091882,22.205144
142,2026-07-31,747.030029,4.215426,8.530708,19.495514
143,2026-08-31,767.049988,1.658483,12.410724,20.230848
144,2026-09-30,770.190002,3.136171,18.733921,16.570739


## 18. Aylık ML özellik fonksiyonları

Her şirket için aylık momentum, oynaklık, maksimum düşüş, hacim değişimi ve sonraki 12 aylık getiri hedefini üreten fonksiyonları tanımlar.


In [18]:
def trailing_max_drawdown(values):
    values = np.asarray(values, dtype=float)

    running_max = np.maximum.accumulate(values)
    drawdowns = values / running_max - 1

    return drawdowns.min() * 100


def create_monthly_features(
    ticker,
    historical_data
):
    try:
        ticker_daily = historical_data[ticker].copy()

        ticker_daily = ticker_daily.dropna(
            subset=["Close"]
        )

        if len(ticker_daily) < 500:
            return pd.DataFrame()

        monthly = pd.DataFrame()

        monthly["close"] = (
            ticker_daily["Close"]
            .resample("ME")
            .last()
        )

        monthly["volume"] = (
            ticker_daily["Volume"]
            .resample("ME")
            .mean()
        )

        monthly["return_1m_pct"] = (
            monthly["close"].pct_change(1) * 100
        )

        monthly["return_3m_pct"] = (
            monthly["close"].pct_change(3) * 100
        )

        monthly["return_6m_pct"] = (
            monthly["close"].pct_change(6) * 100
        )

        monthly["return_12m_pct"] = (
            monthly["close"].pct_change(12) * 100
        )

        monthly_returns = (
            monthly["close"].pct_change()
        )

        monthly["volatility_12m_pct"] = (
            monthly_returns
            .rolling(12)
            .std()
            * np.sqrt(12)
            * 100
        )

        monthly["max_drawdown_12m_pct"] = (
            monthly["close"]
            .rolling(12)
            .apply(
                trailing_max_drawdown,
                raw=True
            )
        )

        monthly["volume_change_3m_pct"] = (
            monthly["volume"]
            / monthly["volume"].shift(3)
            - 1
        ) * 100

        # Modelin tahmin edeceği değer
        monthly["target_return_12m_pct"] = (
            monthly["close"].shift(-12)
            / monthly["close"]
            - 1
        ) * 100

        monthly["yf_ticker"] = ticker
        monthly["date"] = monthly.index

        monthly = monthly.reset_index(drop=True)

        return monthly

    except Exception as error:
        print(ticker, "hatası:", error)
        return pd.DataFrame()


## 19. Makine öğrenmesi eğitim tablosu

Şirket bazındaki aylık özellikleri tek tabloda birleştirir; SPY ve sektör bilgilerini ekler. Model özelliklerini ve ilk hedef değişkenini tanımlar.


In [19]:
training_frames = []
failed_training_tickers = []

for ticker in tqdm(training_tickers):
    ticker_features = create_monthly_features(
        ticker,
        historical_prices
    )

    if ticker_features.empty:
        failed_training_tickers.append(ticker)
    else:
        training_frames.append(ticker_features)

ml_dataset = pd.concat(
    training_frames,
    ignore_index=True
)

print("Eğitim satırı:", len(ml_dataset))
print(
    "Veri üretilemeyen hisseler:",
    failed_training_tickers
)

ml_dataset = ml_dataset.merge(
    spy_monthly,
    on="date",
    how="left"
)

ml_dataset["relative_return_sp500_pct"] = (
    ml_dataset["return_12m_pct"]
    - ml_dataset["spy_return_12m_pct"]
)

company_lookup = (
    sp500_snapshot[[
        "yf_ticker",
        "company_name",
        "sector_sp500"
    ]]
    .drop_duplicates("yf_ticker")
)

ml_dataset = ml_dataset.merge(
    company_lookup,
    on="yf_ticker",
    how="left"
)

feature_columns = [
    "return_1m_pct",
    "return_3m_pct",
    "return_6m_pct",
    "return_12m_pct",
    "relative_return_sp500_pct",
    "volatility_12m_pct",
    "max_drawdown_12m_pct",
    "volume_change_3m_pct",
    "spy_return_3m_pct",
    "spy_return_6m_pct",
    "spy_return_12m_pct"
]

target_column = "target_return_12m_pct"

ml_dataset[feature_columns] = (
    ml_dataset[feature_columns]
    .replace([np.inf, -np.inf], np.nan)
)

model_data = ml_dataset.dropna(
    subset=[target_column]
).copy()

print("Modele uygun satır:", len(model_data))
print("İlk tarih:", model_data["date"].min())
print("Son tarih:", model_data["date"].max())

display(
    model_data[
        [
            "yf_ticker",
            "date"
        ]
        + feature_columns
        + [target_column]
    ].tail()
)


  0%|          | 0/503 [00:00<?, ?it/s]

Eğitim satırı: 70137
Veri üretilemeyen hisseler: ['FDXF', 'HONA', 'Q', 'SNDK']
Modele uygun satır: 64149
İlk tarih: 2014-09-30 00:00:00
Son tarih: 2025-09-30 00:00:00


,yf_ticker,date,return_1m_pct,return_3m_pct,return_6m_pct,return_12m_pct,relative_return_sp500_pct,volatility_12m_pct,max_drawdown_12m_pct,volume_change_3m_pct,spy_return_3m_pct,spy_return_6m_pct,spy_return_12m_pct,target_return_12m_pct
70120,ZTS,2025-05-31,7.819704,1.171392,-3.160884,0.564624,-12.618087,18.278359,-19.247022,-17.848068,-0.507753,-1.559587,13.182711,-53.213016
70121,ZTS,2025-06-30,-7.519421,-4.964315,-3.670706,-9.035878,-23.979351,19.630847,-19.479359,16.625446,10.777209,6.050465,14.943473,-53.205308
70122,ZTS,2025-07-31,-6.202317,-6.472186,-14.118722,-18.038407,-34.222333,19.621409,-24.473505,5.953862,14.319737,5.655474,16.183926,-45.967198
70123,ZTS,2025-08-31,7.277590,-6.942440,-5.852372,-13.726779,-29.587592,21.299473,-24.473505,-1.869760,9.767206,9.209860,15.860813,-49.756540
70124,ZTS,2025-09-30,-6.445008,-5.861324,-10.534664,-24.198935,-41.718281,20.208387,-17.655888,-4.242046,8.121220,19.773671,17.519346,-47.193828


## 20. Rev 2 hedefleri ve zaman ayrımı

Bu bölüm iki ayrı hedef oluşturur. Regresyon modeli hissenin gelecek 12 aylık **toplam getirisini** tahmin eder; bu çıktı güncel fiyatla çarpılarak doğrudan ML hedef fiyatına çevrilir. Sınıflandırma modeli ise hissenin aynı dönemde S&P 500'ü geçip geçmediğini tahmin etmeye devam eder. Eğitim, doğrulama ve test dönemleri kronolojik ayrılır; 12 aylık hedeflerin dönemler arasında taşmasını azaltmak için aralarda boşluk bırakılır.


In [20]:
# SPY'ın gelecek 12 aylık getirisini hesapla
spy_monthly["spy_future_return_12m_pct"] = (
    spy_monthly["spy_close"].shift(-12)
    / spy_monthly["spy_close"]
    - 1
) * 100

ml_dataset = ml_dataset.drop(
    columns=["spy_future_return_12m_pct"],
    errors="ignore"
)

ml_dataset = ml_dataset.merge(
    spy_monthly[["date", "spy_future_return_12m_pct"]],
    on="date",
    how="left"
)

# Rev 2 regresyon hedefi: hissenin doğrudan gelecek 12 aylık getirisi
ml_dataset["target_stock_return_12m_pct"] = (
    ml_dataset["target_return_12m_pct"]
)

# Ayrı sınıflandırma hedefi: S&P 500'e göre fazla getiri ve geçme durumu
ml_dataset["target_excess_return_12m_pct"] = (
    ml_dataset["target_stock_return_12m_pct"]
    - ml_dataset["spy_future_return_12m_pct"]
)

ml_dataset["target_outperform_sp500"] = (
    ml_dataset["target_excess_return_12m_pct"] > 0
).astype(int)

model_data_v2 = ml_dataset.dropna(
    subset=[
        "target_stock_return_12m_pct",
        "target_outperform_sp500"
    ]
).copy()

train_v2 = model_data_v2[model_data_v2["date"] <= "2021-12-31"].copy()
validation_v2 = model_data_v2[
    (model_data_v2["date"] >= "2023-01-01")
    & (model_data_v2["date"] <= "2023-12-31")
].copy()
test_v2 = model_data_v2[model_data_v2["date"] >= "2025-01-01"].copy()

print("Eğitim:", len(train_v2))
print("Doğrulama:", len(validation_v2))
print("Test:", len(test_v2))

# Aşırı getirilerin modeli bozmasını sınırlamak için yalnızca eğitim dönemi sınırları
direct_target = "target_stock_return_12m_pct"
lower_limit = train_v2[direct_target].quantile(0.01)
upper_limit = train_v2[direct_target].quantile(0.99)

for frame in (train_v2, validation_v2, test_v2):
    frame["target_clipped"] = frame[direct_target].clip(lower_limit, upper_limit)

print("Doğrudan getiri alt sınırı:", round(lower_limit, 2))
print("Doğrudan getiri üst sınırı:", round(upper_limit, 2))


Eğitim: 41820
Doğrulama: 5939
Test: 4491
Doğrudan getiri alt sınırı: -49.65
Doğrudan getiri üst sınırı: 136.72


## 21. Regresyon modelleri

Sayısal alanları medyanla tamamlayan, sektörü one-hot biçiminde kodlayan Ridge ve Gradient Boosting modellerini kurar. Regresyon çıktısı kesin getiri vaadi değil, sıralama sinyalidir.


In [21]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import (
    HistGradientBoostingRegressor
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


numeric_features = feature_columns
categorical_features = ["sector_sp500"]


numeric_processor = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])


categorical_processor = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])


preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_processor,
        numeric_features
    ),
    (
        "categorical",
        categorical_processor,
        categorical_features
    )
])


ridge_v2 = Pipeline([
    ("preprocessor", clone(preprocessor)),
    (
        "model",
        Ridge(alpha=20)
    )
])


gradient_v2 = Pipeline([
    ("preprocessor", clone(preprocessor)),
    (
        "model",
        HistGradientBoostingRegressor(
            learning_rate=0.04,
            max_iter=400,
            max_leaf_nodes=20,
            min_samples_leaf=50,
            l2_regularization=5,
            random_state=42
        )
    )
])


regression_models_v2 = {
    "Ridge Direct 12M Return": ridge_v2,
    "Gradient Boosting Direct 12M Return": gradient_v2
}


## 22. Rev 2 regresyon doğrulaması ve fiyat sıralaması

Ridge ve Gradient Boosting modelleri gelecek 12 aylık doğrudan hisse getirisini tahmin eder. MAE/RMSE yüzde puan cinsindedir. Yön doğruluğu, modelin yükseliş veya düşüş yönünü doğru tahmin edip etmediğini gösterir. Ayrıca tahmin gruplarının gerçek getirileri karşılaştırılarak modelin şirketleri doğru sıralayıp sıralamadığı kontrol edilir.


In [22]:
X_train_v2 = train_v2[numeric_features + categorical_features]
y_train_v2 = train_v2["target_clipped"]
X_validation_v2 = validation_v2[numeric_features + categorical_features]
y_validation_v2 = validation_v2["target_clipped"]

validation_results_v2 = []

for model_name, model in regression_models_v2.items():
    model.fit(X_train_v2, y_train_v2)
    predictions = model.predict(X_validation_v2)
    validation_results_v2.append({
        "model": model_name,
        "MAE": mean_absolute_error(y_validation_v2, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_validation_v2, predictions)),
        "R2": r2_score(y_validation_v2, predictions),
        "direction_accuracy_pct": ((predictions > 0) == (y_validation_v2 > 0)).mean() * 100
    })

validation_results_v2 = pd.DataFrame(validation_results_v2).round(2)

baseline_return = y_train_v2.median()
baseline_predictions = np.full(len(y_validation_v2), baseline_return)
baseline_v2 = pd.DataFrame([{
    "model": "Baseline - Median Direct Return",
    "MAE": mean_absolute_error(y_validation_v2, baseline_predictions),
    "RMSE": np.sqrt(mean_squared_error(y_validation_v2, baseline_predictions)),
    "R2": r2_score(y_validation_v2, baseline_predictions),
    "direction_accuracy_pct": ((baseline_predictions > 0) == (y_validation_v2 > 0)).mean() * 100
}]).round(2)

display(pd.concat([validation_results_v2, baseline_v2], ignore_index=True))

gradient_validation_prediction = gradient_v2.predict(X_validation_v2)
ranking_test = validation_v2[["yf_ticker", "date", "target_stock_return_12m_pct"]].copy()
ranking_test["predicted_stock_return_12m_pct"] = gradient_validation_prediction
ranking_test["prediction_group"] = pd.qcut(
    ranking_test["predicted_stock_return_12m_pct"],
    q=5,
    labels=["1 - Lowest", "2", "3", "4", "5 - Highest"],
    duplicates="drop"
)

ranking_result = (
    ranking_test.groupby("prediction_group", observed=True)
    .agg(
        company_month_count=("yf_ticker", "count"),
        predicted_return_mean=("predicted_stock_return_12m_pct", "mean"),
        actual_return_mean=("target_stock_return_12m_pct", "mean"),
        positive_return_rate=("target_stock_return_12m_pct", lambda values: (values > 0).mean() * 100)
    )
    .reset_index()
    .round(2)
)

display(ranking_result)


,model,MAE,RMSE,R2,direction_accuracy_pct
0,Ridge Direct 12M Return,26.08,34.05,-0.01,77.59
1,Gradient Boosting Direct 12M Return,25.30,33.97,-0.01,77.42
2,Baseline - Median Direct Return,25.98,35.98,-0.13,78.11


,prediction_group,company_month_count,predicted_return_mean,actual_return_mean,positive_return_rate
0,1 - Lowest,1188,5.28,23.40,75.42
1,2,1188,14.31,18.71,74.75
2,3,1187,20.30,23.29,77.42
3,4,1188,27.18,28.55,81.06
4,5 - Highest,1188,42.77,46.31,81.90


## 23. Sınıflandırma modeli ve karar eşiği

Bir hissenin S&P 500'ü geçip geçmeyeceğini tahmin eden iki sınıflandırıcıyı karşılaştırır. Gradient Boosting olasılıkları üzerinde doğrulama dönemindeki en iyi dengeli doğruluk eşiğini seçer; bu eşik test verisine bakılmadan belirlenir.


In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    HistGradientBoostingClassifier
)
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score
)


logistic_classifier = Pipeline([
    ("preprocessor", clone(preprocessor)),
    (
        "model",
        LogisticRegression(
            C=0.5,
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    )
])


gradient_classifier = Pipeline([
    ("preprocessor", clone(preprocessor)),
    (
        "model",
        HistGradientBoostingClassifier(
            learning_rate=0.04,
            max_iter=300,
            max_leaf_nodes=20,
            min_samples_leaf=50,
            l2_regularization=5,
            random_state=42
        )
    )
])


classification_models = {
    "Logistic Regression": logistic_classifier,
    "Gradient Boosting Classifier": gradient_classifier
}


X_train_classifier = train_v2[
    numeric_features + categorical_features
]

y_train_classifier = train_v2[
    "target_outperform_sp500"
]


X_validation_classifier = validation_v2[
    numeric_features + categorical_features
]

y_validation_classifier = validation_v2[
    "target_outperform_sp500"
]

classification_results = []

for model_name, model in classification_models.items():
    model.fit(
        X_train_classifier,
        y_train_classifier
    )

    predicted_class = model.predict(
        X_validation_classifier
    )

    predicted_probability = (
        model.predict_proba(
            X_validation_classifier
        )[:, 1]
    )

    classification_results.append({
        "model": model_name,

        "accuracy_pct": (
            accuracy_score(
                y_validation_classifier,
                predicted_class
            ) * 100
        ),

        "balanced_accuracy_pct": (
            balanced_accuracy_score(
                y_validation_classifier,
                predicted_class
            ) * 100
        ),

        "precision_pct": (
            precision_score(
                y_validation_classifier,
                predicted_class,
                zero_division=0
            ) * 100
        ),

        "recall_pct": (
            recall_score(
                y_validation_classifier,
                predicted_class,
                zero_division=0
            ) * 100
        ),

        "roc_auc": roc_auc_score(
            y_validation_classifier,
            predicted_probability
        )
    })


classification_results_df = pd.DataFrame(
    classification_results
).round(2)

display(classification_results_df)

majority_class = (
    y_train_classifier.mode().iloc[0]
)

baseline_class_prediction = np.full(
    len(y_validation_classifier),
    majority_class
)

baseline_accuracy = (
    accuracy_score(
        y_validation_classifier,
        baseline_class_prediction
    ) * 100
)

print(
    "Validation döneminde S&P 500'ü geçenlerin oranı:",
    round(
        y_validation_classifier.mean() * 100,
        2
    ),
    "%"
)

print(
    "Çoğunluk sınıfı baseline doğruluğu:",
    round(baseline_accuracy, 2),
    "%"
)

from sklearn.metrics import balanced_accuracy_score

# Modeli yalnızca eğitim verisiyle yeniden eğit
gradient_classifier.fit(
    X_train_classifier,
    y_train_classifier
)

validation_probabilities = (
    gradient_classifier.predict_proba(
        X_validation_classifier
    )[:, 1]
)

threshold_results = []

for threshold in np.arange(
    0.25,
    0.76,
    0.01
):
    predictions = (
        validation_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": threshold,

        "accuracy_pct": (
            accuracy_score(
                y_validation_classifier,
                predictions
            ) * 100
        ),

        "balanced_accuracy_pct": (
            balanced_accuracy_score(
                y_validation_classifier,
                predictions
            ) * 100
        )
    })

threshold_results_df = pd.DataFrame(
    threshold_results
)

best_threshold_row = (
    threshold_results_df
    .sort_values(
        [
            "balanced_accuracy_pct",
            "accuracy_pct"
        ],
        ascending=False
    )
    .iloc[0]
)

best_classification_threshold = (
    best_threshold_row["threshold"]
)

print(
    "En iyi olasılık eşiği:",
    round(best_classification_threshold, 2)
)

display(
    threshold_results_df.sort_values(
        "balanced_accuracy_pct",
        ascending=False
    ).head(10).round(2)
)


,model,accuracy_pct,balanced_accuracy_pct,precision_pct,recall_pct,roc_auc
0,Logistic Regression,53.91,54.53,43.70,57.50,0.57
1,Gradient Boosting Classifier,54.23,55.55,44.35,61.84,0.59


Validation döneminde S&P 500'ü geçenlerin oranı: 39.54 %
Çoğunluk sınıfı baseline doğruluğu: 39.54 %
En iyi olasılık eşiği: 0.55


,threshold,accuracy_pct,balanced_accuracy_pct
30,0.55,59.00,56.41
29,0.54,58.04,56.19
31,0.56,59.49,56.17
33,0.58,60.60,56.06
27,0.52,56.29,55.97
26,0.51,55.40,55.86
28,0.53,56.93,55.84
34,0.59,60.89,55.82
32,0.57,59.69,55.74
35,0.60,61.26,55.64


## 24. Dokunulmamış test döneminde Rev 2 değerlendirmesi

Regresyon modeli doğrudan 12 aylık getiriyi; sınıflandırma modeli ise S&P 500'ü geçme olasılığını üretir. Test dönemi model seçimi ve eşik ayarında kullanılmamıştır. Bu hücre hem fiyat tahmin hatasını hem yön doğruluğunu hem de sıralama ilişkisini ölçer.


In [24]:
train_validation_v2 = pd.concat([train_v2, validation_v2], ignore_index=True)
X_train_final = train_validation_v2[numeric_features + categorical_features]
y_train_regression_final = train_validation_v2["target_clipped"]
y_train_classification_final = train_validation_v2["target_outperform_sp500"]

final_regression_model_v2 = gradient_v2
final_regression_model_v2.fit(X_train_final, y_train_regression_final)

final_classifier_model = gradient_classifier
final_classifier_model.fit(X_train_final, y_train_classification_final)

X_test_final = test_v2[numeric_features + categorical_features]
test_return_prediction = final_regression_model_v2.predict(X_test_final)
test_outperform_probability = final_classifier_model.predict_proba(X_test_final)[:, 1]
test_class_prediction = (test_outperform_probability >= best_classification_threshold).astype(int)

from scipy.stats import spearmanr

test_actual_return = test_v2["target_stock_return_12m_pct"]
test_actual_clipped = test_v2["target_clipped"]
test_actual_class = test_v2["target_outperform_sp500"]
spearman_correlation, spearman_p_value = spearmanr(
    test_return_prediction, test_actual_return, nan_policy="omit"
)

test_evaluation = pd.DataFrame([{
    "regression_MAE_pct_point": mean_absolute_error(test_actual_clipped, test_return_prediction),
    "regression_RMSE_pct_point": np.sqrt(mean_squared_error(test_actual_clipped, test_return_prediction)),
    "regression_R2": r2_score(test_actual_clipped, test_return_prediction),
    "price_direction_accuracy_pct": ((test_return_prediction > 0) == (test_actual_return > 0)).mean() * 100,
    "outperform_direction_accuracy_pct": accuracy_score(test_actual_class, test_class_prediction) * 100,
    "balanced_accuracy_pct": balanced_accuracy_score(test_actual_class, test_class_prediction) * 100,
    "roc_auc": roc_auc_score(test_actual_class, test_outperform_probability),
    "spearman_rank_correlation": spearman_correlation,
    "spearman_p_value": spearman_p_value
}]).round(3)

display(test_evaluation)

test_ranking = test_v2[["yf_ticker", "date", "target_stock_return_12m_pct", "target_outperform_sp500"]].copy()
test_ranking["predicted_stock_return_12m_pct"] = test_return_prediction
test_ranking["outperform_probability_pct"] = test_outperform_probability * 100
test_ranking["prediction_group"] = pd.qcut(
    test_ranking["predicted_stock_return_12m_pct"],
    q=5,
    labels=["1 - Lowest", "2", "3", "4", "5 - Highest"],
    duplicates="drop"
)

test_ranking_result = (
    test_ranking.groupby("prediction_group", observed=True)
    .agg(
        observation_count=("yf_ticker", "count"),
        predicted_return_mean=("predicted_stock_return_12m_pct", "mean"),
        actual_return_mean=("target_stock_return_12m_pct", "mean"),
        positive_return_rate=("target_stock_return_12m_pct", lambda x: (x > 0).mean() * 100),
        outperform_rate=("target_outperform_sp500", "mean"),
        average_model_probability=("outperform_probability_pct", "mean")
    )
    .reset_index()
)
test_ranking_result["outperform_rate"] *= 100
display(test_ranking_result.round(2))


,regression_MAE_pct_point,regression_RMSE_pct_point,regression_R2,price_direction_accuracy_pct,outperform_direction_accuracy_pct,balanced_accuracy_pct,roc_auc,spearman_rank_correlation,spearman_p_value
0,30.259,40.248,-0.026,64.395,60.499,54.056,0.562,0.058,0.0


,prediction_group,observation_count,predicted_return_mean,actual_return_mean,positive_return_rate,outperform_rate,average_model_probability
0,1 - Lowest,899,6.42,18.04,68.85,34.26,40.90
1,2,898,14.27,13.20,63.92,32.18,43.47
2,3,898,18.72,17.62,68.15,40.65,46.40
3,4,898,24.48,26.39,62.14,39.31,50.05
4,5 - Highest,898,38.55,50.15,67.26,44.43,56.47


## 25. Güncel şirketler için Rev 2 hedef fiyatları

Final regresyon modeli tüm etiketli geçmiş üzerinde yeniden eğitilir ve güncel şirketler için 12 aylık toplam getiri tahmini üretir. Bu tahmin güncel fiyatla çarpılarak doğrudan `ml_target_price_12m` oluşturulur. Sınıflandırma modeli ayrıca S&P 500'ü geçme olasılığını üretir. İlk hedefin daha sonra ezilmemesi için tahmin tarihi, başlangıç fiyatı ve vade tarihi snapshot'a eklenir.


In [25]:
all_labeled_data = ml_dataset.dropna(
    subset=["target_stock_return_12m_pct", "target_outperform_sp500"]
).copy()
all_labeled_data["target_clipped"] = all_labeled_data["target_stock_return_12m_pct"].clip(
    lower=lower_limit, upper=upper_limit
)

X_all_labeled = all_labeled_data[numeric_features + categorical_features]
final_regression_model_v2.fit(X_all_labeled, all_labeled_data["target_clipped"])
final_classifier_model.fit(X_all_labeled, all_labeled_data["target_outperform_sp500"])
print("Rev 2 final modelleri eğitildi.")

latest_ml_features = (
    ml_dataset.sort_values("date")
    .groupby("yf_ticker", as_index=False)
    .tail(1)
    .copy()
)
latest_ml_features = latest_ml_features.dropna(subset=numeric_features, thresh=7)
X_current = latest_ml_features[numeric_features + categorical_features]

latest_ml_features["ml_predicted_return_12m_pct"] = final_regression_model_v2.predict(X_current)
latest_ml_features["ml_outperform_probability_pct"] = final_classifier_model.predict_proba(X_current)[:, 1] * 100

# Model uç değerlerini eğitim sınırlarıyla sınırlı tut
latest_ml_features["ml_predicted_return_12m_pct"] = latest_ml_features[
    "ml_predicted_return_12m_pct"
].clip(lower=lower_limit, upper=upper_limit)

latest_ml_features["ml_rank_percentile"] = (
    latest_ml_features["ml_predicted_return_12m_pct"].rank(pct=True) * 100
)
latest_ml_features["ml_outperform_prediction"] = (
    latest_ml_features["ml_outperform_probability_pct"]
    >= best_classification_threshold * 100
)
latest_ml_features["ml_signal"] = np.select(
    [
        (latest_ml_features["ml_rank_percentile"] >= 80)
        & (latest_ml_features["ml_outperform_probability_pct"] >= 57),
        latest_ml_features["ml_rank_percentile"] <= 20
    ],
    ["STRONG", "WEAK"],
    default="NEUTRAL"
)

forecast_date = pd.Timestamp.utcnow().tz_localize(None).normalize()
latest_ml_features["ml_forecast_date"] = forecast_date
latest_ml_features["ml_maturity_date"] = forecast_date + pd.DateOffset(months=12)

ml_output_columns = [
    "yf_ticker", "date", "ml_predicted_return_12m_pct",
    "ml_outperform_probability_pct", "ml_outperform_prediction",
    "ml_rank_percentile", "ml_signal", "ml_forecast_date", "ml_maturity_date"
]
ml_current_output = latest_ml_features[ml_output_columns].rename(columns={"date": "ml_data_date"})

sp500_snapshot = sp500_snapshot.drop(
    columns=[c for c in ml_current_output.columns if c != "yf_ticker" and c in sp500_snapshot.columns],
    errors="ignore"
).merge(ml_current_output, on="yf_ticker", how="left", validate="one_to_one")

sp500_snapshot["ml_start_price"] = pd.to_numeric(sp500_snapshot["current_price"], errors="coerce")
sp500_snapshot["ml_target_price_12m"] = (
    sp500_snapshot["ml_start_price"]
    * (1 + sp500_snapshot["ml_predicted_return_12m_pct"] / 100)
)

round_columns = [
    "ml_predicted_return_12m_pct", "ml_target_price_12m",
    "ml_outperform_probability_pct", "ml_rank_percentile"
]
sp500_snapshot[round_columns] = sp500_snapshot[round_columns].round(2)

print("Tahmin üretilebilecek şirket:", len(latest_ml_features))
display(sp500_snapshot[[
    "ticker", "current_price", "ml_target_price_12m",
    "ml_predicted_return_12m_pct", "ml_outperform_probability_pct",
    "ml_forecast_date", "ml_maturity_date"
]].sort_values("ml_predicted_return_12m_pct", ascending=False).head(30))


Rev 2 final modelleri eğitildi.
Tahmin üretilebilecek şirket: 499


,ticker,current_price,ml_target_price_12m,ml_predicted_return_12m_pct,ml_outperform_probability_pct,ml_forecast_date,ml_maturity_date
126,GLW,154.30,270.57,75.36,68.70,2026-09-07,2027-09-07
4,ACN,186.72,323.80,73.42,71.24,2026-09-07,2027-09-07
444,TTD,14.43,25.01,73.34,60.51,2026-09-07,2027-09-07
490,WDC,467.46,810.32,73.34,60.63,2026-09-07,2027-09-07
381,QCOM,168.74,290.01,71.87,59.52,2026-09-07,2027-09-07
300,MRVL,223.55,384.08,71.81,63.39,2026-09-07,2027-09-07
199,FLEX,109.51,188.05,71.72,62.54,2026-09-07,2027-09-07
406,STX,849.28,1454.92,71.31,69.57,2026-09-07,2027-09-07
6,AMD,477.57,817.94,71.27,67.55,2026-09-07,2027-09-07
129,CSGP,30.91,51.79,67.55,63.08,2026-09-07,2027-09-07


## 26. Puanlama yardımcıları

Farklı ölçeklerdeki göstergeleri 0–100 yüzdelik puanlara dönüştüren genel ve sektör içi sıralama fonksiyonlarını tanımlar.


In [26]:
import numpy as np
import pandas as pd


def percentile_score(series, higher_is_better=True):
    numeric = pd.to_numeric(
        series,
        errors="coerce"
    )

    score = numeric.rank(
        pct=True,
        method="average"
    ) * 100

    if not higher_is_better:
        score = 100 - score

    return score


def sector_percentile_score(
    dataframe,
    column,
    higher_is_better=True
):
    numeric = pd.to_numeric(
        dataframe[column],
        errors="coerce"
    )

    score = numeric.groupby(
        dataframe["sector_sp500"]
    ).rank(
        pct=True,
        method="average"
    ) * 100

    if not higher_is_better:
        score = 100 - score

    return score


def row_average(dataframe, columns):
    return dataframe[columns].mean(
        axis=1,
        skipna=True
    )


## 27. Alt puanlar ve nihai puan

Temel analiz, analist, ML, finansal kalite, tutarlılık ve risk güvenliği alt puanlarını hesaplar. Veri eksikliği nötr puanla ele alınır ve nihai puana veri tamlığı cezası uygulanır.


In [27]:
score_df = sp500_snapshot.copy()


score_df["fundamental_upside_score"] = (
    percentile_score(
        score_df["fundamental_upside_pct"],
        higher_is_better=True
    )
)


score_df["valuation_method_score"] = (
    pd.to_numeric(
        score_df["valuation_method_count"],
        errors="coerce"
    )
    .clip(lower=0, upper=3)
    / 3
    * 100
)


score_df["fcf_yield_score"] = (
    sector_percentile_score(
        score_df,
        "fcf_yield_pct",
        higher_is_better=True
    )
)


score_df["fundamental_score"] = (
    score_df["fundamental_upside_score"] * 0.60
    + score_df["valuation_method_score"] * 0.20
    + score_df["fcf_yield_score"].fillna(50) * 0.20
)

score_df["analyst_upside_score"] = (
    percentile_score(
        score_df["analyst_upside_pct"],
        higher_is_better=True
    )
)


score_df["analyst_count_score"] = (
    percentile_score(
        score_df["analyst_count"],
        higher_is_better=True
    )
)


score_df["analyst_dispersion_score"] = (
    percentile_score(
        score_df["analyst_dispersion_pct"],
        higher_is_better=False
    )
)


score_df["analyst_score"] = (
    score_df["analyst_upside_score"] * 0.60
    + score_df["analyst_count_score"].fillna(50) * 0.20
    + score_df["analyst_dispersion_score"].fillna(50) * 0.20
)

score_df["ml_score"] = (
    pd.to_numeric(
        score_df["ml_rank_percentile"],
        errors="coerce"
    ) * 0.60
    + pd.to_numeric(
        score_df["ml_outperform_probability_pct"],
        errors="coerce"
    ) * 0.40
)

quality_metrics = {
    "revenue_growth_score": (
        "revenue_growth_pct",
        True
    ),
    "earnings_growth_score": (
        "earnings_growth_pct",
        True
    ),
    "ebitda_margin_score": (
        "ebitda_margin_pct",
        True
    ),
    "net_margin_score": (
        "net_margin_pct",
        True
    ),
    "roe_score": (
        "roe_pct",
        True
    ),
    "fcf_quality_score": (
        "fcf_yield_pct",
        True
    )
}


quality_score_columns = []

for score_column, (
    source_column,
    higher_is_better
) in quality_metrics.items():

    score_df[score_column] = (
        sector_percentile_score(
            score_df,
            source_column,
            higher_is_better
        )
    )

    quality_score_columns.append(
        score_column
    )


score_df["quality_score"] = row_average(
    score_df,
    quality_score_columns
)

score_df["volatility_safety_score"] = (
    percentile_score(
        score_df["volatility_12m_pct"],
        higher_is_better=False
    )
)


# Drawdown değerleri negatif olduğu için
# -10 değeri -50'den daha iyidir.
score_df["drawdown_safety_score"] = (
    percentile_score(
        score_df["max_drawdown_12m_pct"],
        higher_is_better=True
    )
)


score_df["debt_safety_score"] = (
    sector_percentile_score(
        score_df,
        "net_debt_ebitda",
        higher_is_better=False
    )
)


score_df["analyst_uncertainty_safety_score"] = (
    percentile_score(
        score_df["analyst_dispersion_pct"],
        higher_is_better=False
    )
)


risk_columns = [
    "volatility_safety_score",
    "drawdown_safety_score",
    "debt_safety_score",
    "analyst_uncertainty_safety_score"
]


score_df["risk_safety_score"] = row_average(
    score_df,
    risk_columns
)

required_score_inputs = [
    "fundamental_upside_pct",
    "valuation_method_count",
    "fcf_yield_pct",

    "analyst_upside_pct",
    "analyst_count",
    "analyst_dispersion_pct",

    "ml_rank_percentile",
    "ml_outperform_probability_pct",

    "revenue_growth_pct",
    "earnings_growth_pct",
    "ebitda_margin_pct",
    "net_margin_pct",
    "roe_pct",

    "volatility_12m_pct",
    "max_drawdown_12m_pct",
    "net_debt_ebitda"
]


score_df["data_completeness_pct"] = (
    score_df[
        required_score_inputs
    ].notna().mean(axis=1) * 100
)

main_score_columns = [
    "fundamental_score",
    "analyst_score",
    "ml_score",
    "quality_score",
    "consistency_score",
    "risk_safety_score"
]


# Eksik ana puanları geçici olarak nötr kabul et
for column in main_score_columns:
    score_df[column] = (
        pd.to_numeric(
            score_df[column],
            errors="coerce"
        ).fillna(50)
    )


score_df["raw_final_score"] = (
    score_df["fundamental_score"] * 0.25
    + score_df["analyst_score"] * 0.20
    + score_df["ml_score"] * 0.25
    + score_df["quality_score"] * 0.15
    + score_df["consistency_score"] * 0.10
    + score_df["risk_safety_score"] * 0.05
)


# Eksik verisi fazla olan şirketlere küçük güven cezası
score_df["final_score"] = (
    score_df["raw_final_score"]
    * (
        0.70
        + 0.30
        * score_df["data_completeness_pct"]
        / 100
    )
).round(2)

sp500_snapshot = score_df.copy()

print(
    "Puanlama tamamlandı:",
    sp500_snapshot.shape
)


Puanlama tamamlandı: (503, 124)


## 28. S&P 500 göreceli sıralaması

Nihai puanı S&P 500 içindeki yüzdelik sıraya çevirir. Üst %5 güçlü aday, sonraki %15 olumlu, orta %50 nötr, sonraki %20 zayıf ve alt %10 yüksek risk/zayıf olarak etiketlenir. Güçlü adaylar ayrıca veri tamlığı, yöntem sayısı ve yön uzlaşısı koşullarından geçmelidir.


In [28]:
sp500_snapshot["final_rank_percentile"] = (
    sp500_snapshot["final_score"]
    .rank(
        pct=True,
        method="average"
    )
    * 100
).round(2)


sp500_snapshot["sp500_rank"] = (
    sp500_snapshot["final_score"]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)

sp500_snapshot["final_category"] = pd.cut(
    sp500_snapshot["final_rank_percentile"],
    bins=[
        -np.inf,
        10,
        30,
        80,
        95,
        np.inf
    ],
    labels=[
        "Yüksek Risk / Zayıf",
        "Zayıf",
        "Nötr",
        "Olumlu",
        "Güçlü Aday"
    ],
    include_lowest=True
)

strong_conditions = (
    (sp500_snapshot["final_rank_percentile"] >= 95)
    & (sp500_snapshot["data_completeness_pct"] >= 70)
    & (sp500_snapshot["valuation_method_count"] >= 2)
    & (sp500_snapshot["direction_agreement"] == True)
)


sp500_snapshot["qualified_strong_candidate"] = (
    strong_conditions
)


sp500_snapshot.loc[
    (
        sp500_snapshot["final_category"]
        == "Güçlü Aday"
    )
    & (
        sp500_snapshot[
            "qualified_strong_candidate"
        ] == False
    ),
    "final_category"
] = "Olumlu"


## 29. Türkçe final sonuç tablosu

Ana sonuç kolonlarını Türkçe adlarla sunar, hisseleri S&P 500 sırasına göre listeler ve kategori dağılımını gösterir. Hesaplama kolonları İngilizce bırakıldığı için önceki hücrelerin çalışma düzeni korunur.


In [29]:
ranking_result = sp500_snapshot[[
    "sp500_rank",
    "ticker",
    "company_name",
    "sector_sp500",
    "current_price",

    "fundamental_score",
    "analyst_score",
    "ml_score",
    "quality_score",
    "consistency_score",
    "risk_safety_score",

    "data_completeness_pct",
    "final_score",
    "final_rank_percentile",
    "final_category",
    "qualified_strong_candidate"
]].copy()


ranking_result = ranking_result.rename(
    columns={
        "sp500_rank": "S&P 500 Sırası",
        "ticker": "Hisse Kodu",
        "company_name": "Şirket Adı",
        "sector_sp500": "Sektör",
        "current_price": "Güncel Fiyat",

        "fundamental_score":
            "Temel Analiz Puanı",
        "analyst_score":
            "Analist Puanı",
        "ml_score":
            "ML Puanı",
        "quality_score":
            "Finansal Kalite Puanı",
        "consistency_score":
            "Analist–Temel Analiz Tutarlılığı",
        "risk_safety_score":
            "Risk Güvenliği Puanı",

        "data_completeness_pct":
            "Veri Tamlığı (%)",
        "final_score":
            "Nihai Puan",
        "final_rank_percentile":
            "S&P 500 Yüzdelik Sırası",
        "final_category":
            "Sonuç",
        "qualified_strong_candidate":
            "Güvenlik Şartlarını Geçti mi?"
    }
)


ranking_result[
    "Güvenlik Şartlarını Geçti mi?"
] = ranking_result[
    "Güvenlik Şartlarını Geçti mi?"
].map({
    True: "Evet",
    False: "Hayır"
})


ranking_result = ranking_result.sort_values(
    "S&P 500 Sırası"
).reset_index(drop=True)

display(ranking_result.head(50))

display(
    ranking_result[
        "Sonuç"
    ]
    .value_counts()
    .rename_axis("Kategori")
    .reset_index(name="Şirket Sayısı")
)


,S&P 500 Sırası,Hisse Kodu,Şirket Adı,Sektör,Güncel Fiyat,Temel Analiz Puanı,Analist Puanı,ML Puanı,Finansal Kalite Puanı,Analist–Temel Analiz Tutarlılığı,Risk Güvenliği Puanı,Veri Tamlığı (%),Nihai Puan,S&P 500 Yüzdelik Sırası,Sonuç,Güvenlik Şartlarını Geçti mi?
0,1,ADSK,Autodesk,Information Technology,217.90,84.829995,78.704035,76.934,62.112674,99.95,33.953720,100.00,77.19,100.00,Güçlü Aday,Evet
1,2,MU,Micron Technology,Information Technology,1016.59,68.481690,78.779875,82.530,81.305061,92.78,25.503103,100.00,76.26,99.80,Güçlü Aday,Evet
2,3,ON,ON Semiconductor,Information Technology,74.38,82.982044,78.049832,84.188,42.810739,99.04,23.220619,100.00,74.89,99.60,Güçlü Aday,Evet
3,4,NXPI,NXP Semiconductors,Information Technology,227.84,87.439284,76.003824,75.336,64.509858,80.75,19.925641,100.00,74.64,99.40,Güçlü Aday,Evet
4,5,FSLR,First Solar,Information Technology,204.45,94.342084,74.035496,75.948,56.039716,52.98,30.406773,100.00,72.60,99.20,Güçlü Aday,Evet
5,6,HPE,Hewlett Packard Enterprise,Information Technology,52.00,90.771162,69.213317,80.124,50.426969,66.43,25.583383,100.00,72.05,99.01,Güçlü Aday,Evet
6,7,DECK,Deckers Brands,Consumer Discretionary,85.81,89.861424,73.549214,61.096,63.101178,81.79,35.975847,100.00,71.89,98.81,Güçlü Aday,Evet
7,8,INTU,Intuit,Information Technology,332.70,90.123888,62.493450,85.630,51.414303,65.89,20.550177,100.00,71.77,98.61,Güçlü Aday,Evet
8,9,ALB,Albemarle Corporation,Materials,126.28,93.187251,67.574270,68.186,65.600000,73.40,28.731244,93.75,71.12,98.41,Güçlü Aday,Evet
9,10,BR,Broadridge Financial Solutions,Industrials,172.88,86.725503,58.233913,69.908,63.824422,79.35,46.407042,100.00,70.63,98.21,Güçlü Aday,Evet


,Kategori,Şirket Sayısı
0,Nötr,252
1,Zayıf,100
2,Olumlu,76
3,Yüksek Risk / Zayıf,50
4,Güçlü Aday,25


## Rev 2 — Yorumlama sınırları

- Temel hedef; sektör medyanı F/K, FD/FAVÖK ve serbest nakit akışı yöntemlerinin geçerli olanlarının ortalamasıdır.
- Analist hedefi Yahoo Finance'ın güncel ortalama hedefidir; hedef tarihi ve analist kapsamı kaynak tarafından sınırlanabilir.
- Rev 2 ML hedefi doğrudan 12 aylık getiri tahmininden fiyat üretir. Regresyon performansı mutlaka aşağıdaki test metrikleriyle birlikte yorumlanmalıdır.
- S&P 500'ü geçme olasılığı ayrı sınıflandırma modelinden gelir ve fiyat tahmini değildir.
- İlk ML tahmini daha sonra değiştirilmemeli; her yeni çalışma ayrı bir `forecast_id` ile saklanmalıdır.
- Aylık uyum, beklenen bileşik rota ile gerçek fiyatın ara karşılaştırmasıdır. Nihai doğruluk ancak 12 aylık vade dolduğunda hesaplanır.
- Sonuçlar yatırım tavsiyesi veya getiri garantisi değildir.


## 30. Zaman sıralı portföy backtest'i

Bu bölüm modeli geçmişte gerçekten çalıştırıyormuşuz gibi sınar. Her üç aylık seçim tarihinde model yalnızca o gün itibarıyla sonucu tamamen belli olmuş gözlemlerle eğitilir. Örneğin 2022 Mart seçimi yapılırken, 12 aylık sonucu henüz tamamlanmamış 2021 Nisan ve sonrası eğitim verisine alınmaz.

Testte 20, 30 ve 50 hisselik eşit ağırlıklı portföyler karşılaştırılır. Tek hisse ağırlığı en fazla %5, tek sektör ağırlığı en fazla %25 ve portföy değişim maliyeti %0,10 kabul edilir.

> **Sınır:** Tarihsel analist hedefleri ve tarihsel bilanço snapshot'ları bu notebook'ta bulunmadığı için backtest yalnızca o tarihte üretilebilen fiyat/işlem hacmi tabanlı ML sinyallerini kullanır. Güncel S&P 500 şirket listesinin geçmişe uygulanması ayrıca hayatta kalma yanlılığı doğurabilir. Bu nedenle sonuçlar yatırım garantisi değil, model araştırmasıdır.

In [ ]:
from sklearn.base import clone

BACKTEST_PORTFOLIO_SIZES = [20, 30, 50]
BACKTEST_HOLDING_MONTHS = 3
BACKTEST_TRANSACTION_COST_RATE = 0.001  # %0,10
BACKTEST_MAX_STOCK_WEIGHT = 0.05
BACKTEST_MAX_SECTOR_WEIGHT = 0.25
BACKTEST_MIN_TRAINING_ROWS = 1500
BACKTEST_MIN_TRAINING_COMPANIES = 100


def add_forward_holding_returns(dataset, holding_months=3):
    """Her şirket ve SPY için sonraki elde tutma dönemi getirisini oluşturur."""
    result = dataset.sort_values(["yf_ticker", "date"]).copy()
    result["holding_end_price"] = (
        result.groupby("yf_ticker")["close"].shift(-holding_months)
    )
    result["holding_return_pct"] = (
        result["holding_end_price"] / result["close"] - 1
    ) * 100

    spy_returns = spy_monthly[["date", "spy_close"]].sort_values("date").copy()
    spy_returns["spy_holding_end"] = spy_returns["spy_close"].shift(-holding_months)
    spy_returns["spy_holding_return_pct"] = (
        spy_returns["spy_holding_end"] / spy_returns["spy_close"] - 1
    ) * 100

    return result.merge(
        spy_returns[["date", "spy_holding_return_pct"]],
        on="date",
        how="left"
    )


def select_sector_limited_portfolio(candidates, portfolio_size):
    """Eşit ağırlık altında hisse ve sektör sınırlarına uyan en yüksek skorları seçer."""
    stock_weight = 1 / portfolio_size
    if stock_weight > BACKTEST_MAX_STOCK_WEIGHT + 1e-12:
        raise ValueError("Portföy büyüklüğü hisse başına %5 sınırını aşıyor.")

    max_names_per_sector = max(
        1,
        int(np.floor(BACKTEST_MAX_SECTOR_WEIGHT / stock_weight + 1e-12))
    )
    selected_indices = []
    sector_counts = {}

    for index, row in candidates.sort_values("selection_score", ascending=False).iterrows():
        sector = row.get("sector_sp500")
        sector = "Bilinmiyor" if pd.isna(sector) else str(sector)
        if sector_counts.get(sector, 0) >= max_names_per_sector:
            continue
        selected_indices.append(index)
        sector_counts[sector] = sector_counts.get(sector, 0) + 1
        if len(selected_indices) == portfolio_size:
            break

    return candidates.loc[selected_indices].copy()


backtest_source = add_forward_holding_returns(
    ml_dataset,
    BACKTEST_HOLDING_MONTHS
)

available_months = sorted(pd.to_datetime(backtest_source["date"].dropna().unique()))
rebalance_dates = [
    date for date in available_months
    if date.month in [3, 6, 9, 12]
    and backtest_source.loc[
        backtest_source["date"].eq(date), "holding_return_pct"
    ].notna().sum() >= max(BACKTEST_PORTFOLIO_SIZES)
]

print("Uygun üç aylık seçim tarihi:", len(rebalance_dates))
print("İlk / son seçim tarihi:", rebalance_dates[0], "/", rebalance_dates[-1])

## 31. Her seçim tarihinde modeli yeniden kurma

Model her seçim tarihinde genişleyen geçmiş pencereyle yeniden eğitilir. Eğitim setine yalnızca 12 aylık sonucu seçim tarihinden önce tamamlanmış satırlar girer. Modelin gördüğü gelecek bilgi miktarı böylece sıfırlanır.

Hisseler, tahmini 12 aylık getiri sırası ile S&P 500'ü geçme olasılığının birleşimiyle sıralanır. Bu puan yalnızca portföy seçimi içindir; kesin fiyat garantisi değildir.

In [ ]:
walk_forward_predictions = []
skipped_rebalance_dates = []

for rebalance_date in tqdm(rebalance_dates, desc="Walk-forward dönemleri"):
    information_cutoff = rebalance_date - pd.DateOffset(months=12)

    historical_train = backtest_source[
        (backtest_source["date"] <= information_cutoff)
        & backtest_source["target_stock_return_12m_pct"].notna()
        & backtest_source["target_outperform_sp500"].notna()
    ].copy()

    current_candidates = backtest_source[
        (backtest_source["date"] == rebalance_date)
        & backtest_source["holding_return_pct"].notna()
    ].copy()

    if (
        len(historical_train) < BACKTEST_MIN_TRAINING_ROWS
        or historical_train["yf_ticker"].nunique() < BACKTEST_MIN_TRAINING_COMPANIES
        or len(current_candidates) < max(BACKTEST_PORTFOLIO_SIZES)
    ):
        skipped_rebalance_dates.append(rebalance_date)
        continue

    # Uç değer sınırları her tarihte sadece geçmiş eğitim verisinden hesaplanır.
    train_lower = historical_train["target_stock_return_12m_pct"].quantile(0.01)
    train_upper = historical_train["target_stock_return_12m_pct"].quantile(0.99)
    y_regression = historical_train["target_stock_return_12m_pct"].clip(
        train_lower, train_upper
    )

    period_regressor = clone(gradient_v2)
    period_classifier = clone(gradient_classifier)
    period_regressor.fit(
        historical_train[numeric_features + categorical_features],
        y_regression
    )
    period_classifier.fit(
        historical_train[numeric_features + categorical_features],
        historical_train["target_outperform_sp500"].astype(int)
    )

    X_period = current_candidates[numeric_features + categorical_features]
    current_candidates["predicted_return_12m_pct"] = np.clip(
        period_regressor.predict(X_period), train_lower, train_upper
    )
    current_candidates["outperform_probability_pct"] = (
        period_classifier.predict_proba(X_period)[:, 1] * 100
    )
    current_candidates["predicted_return_rank_pct"] = (
        current_candidates["predicted_return_12m_pct"].rank(pct=True) * 100
    )
    current_candidates["selection_score"] = (
        current_candidates["predicted_return_rank_pct"] * 0.60
        + current_candidates["outperform_probability_pct"] * 0.40
    )
    current_candidates["rebalance_date"] = rebalance_date
    current_candidates["training_cutoff"] = information_cutoff

    walk_forward_predictions.append(current_candidates)

if not walk_forward_predictions:
    raise RuntimeError("Backtest için yeterli walk-forward dönem üretilemedi.")

walk_forward_predictions = pd.concat(walk_forward_predictions, ignore_index=True)
print("Test edilen seçim dönemi:", walk_forward_predictions["rebalance_date"].nunique())
print("Atlanan erken dönem:", len(skipped_rebalance_dates))

## 32. 20/30/50 hisselik portföyleri oluşturma

Her seçim tarihinde en yüksek ML puanlı hisseler alınır. Sektör sınırı nedeniyle bir sektörden aşırı sayıda şirket seçilemez. Portföy değiştikçe tahmini %0,10 işlem maliyeti düşülür. `turnover_pct`, önceki dönemden değişen portföy oranını gösterir.

In [ ]:
portfolio_period_rows = []
portfolio_holdings_rows = []

for portfolio_size in BACKTEST_PORTFOLIO_SIZES:
    previous_tickers = set()

    for rebalance_date, candidates in walk_forward_predictions.groupby("rebalance_date"):
        selected = select_sector_limited_portfolio(candidates, portfolio_size)
        if len(selected) < portfolio_size:
            continue

        current_tickers = set(selected["yf_ticker"])
        overlap = len(previous_tickers.intersection(current_tickers))
        turnover = 1.0 if not previous_tickers else 1 - overlap / portfolio_size
        transaction_cost = turnover * BACKTEST_TRANSACTION_COST_RATE

        gross_return = selected["holding_return_pct"].mean() / 100
        net_return = (1 + gross_return) * (1 - transaction_cost) - 1
        benchmark_return = selected["spy_holding_return_pct"].iloc[0] / 100

        portfolio_period_rows.append({
            "portfolio_size": portfolio_size,
            "rebalance_date": rebalance_date,
            "holding_end_date": rebalance_date + pd.DateOffset(months=BACKTEST_HOLDING_MONTHS),
            "gross_return_pct": gross_return * 100,
            "net_return_pct": net_return * 100,
            "spy_return_pct": benchmark_return * 100,
            "excess_return_pct": (net_return - benchmark_return) * 100,
            "positive_period": net_return > 0,
            "beat_sp500": net_return > benchmark_return,
            "turnover_pct": turnover * 100,
            "transaction_cost_pct": transaction_cost * 100,
            "selected_count": len(selected)
        })

        selected_export = selected[[
            "yf_ticker", "company_name", "sector_sp500", "close",
            "predicted_return_12m_pct", "outperform_probability_pct",
            "selection_score", "holding_return_pct"
        ]].copy()
        selected_export["portfolio_size"] = portfolio_size
        selected_export["rebalance_date"] = rebalance_date
        selected_export["equal_weight_pct"] = 100 / portfolio_size
        portfolio_holdings_rows.append(selected_export)
        previous_tickers = current_tickers

portfolio_periods = pd.DataFrame(portfolio_period_rows).sort_values(
    ["portfolio_size", "rebalance_date"]
).reset_index(drop=True)
portfolio_holdings = pd.concat(portfolio_holdings_rows, ignore_index=True)

display(portfolio_periods.tail(12).round(2))

## 33. Portföy başarı ölçümleri

`Yıllıklandırılmış Getiri`, ardışık üç aylık net getirilerin bileşik sonucudur. `Maksimum Düşüş`, portföyün önceki zirvesinden gördüğü en büyük kaybı; `Sharpe`, risksiz faiz çıkarılmadan oynaklık başına getiriyi gösterir. Ana karar ölçütleri endeksi geçme oranı, yıllıklandırılmış net getiri ve maksimum düşüştür.

In [ ]:
def calculate_max_drawdown_from_returns(return_series):
    wealth = (1 + return_series).cumprod()
    drawdown = wealth / wealth.cummax() - 1
    return drawdown.min() * 100


summary_rows = []
equity_curve_rows = []

for portfolio_size, group in portfolio_periods.groupby("portfolio_size"):
    group = group.sort_values("rebalance_date").copy()
    portfolio_returns = group["net_return_pct"] / 100
    benchmark_returns = group["spy_return_pct"] / 100
    periods_per_year = 12 / BACKTEST_HOLDING_MONTHS
    years = len(group) / periods_per_year

    final_portfolio_value = 10_000 * (1 + portfolio_returns).prod()
    final_benchmark_value = 10_000 * (1 + benchmark_returns).prod()
    annualized_return = (
        (final_portfolio_value / 10_000) ** (1 / years) - 1
    ) if years > 0 else np.nan
    benchmark_annualized = (
        (final_benchmark_value / 10_000) ** (1 / years) - 1
    ) if years > 0 else np.nan
    annualized_volatility = portfolio_returns.std(ddof=1) * np.sqrt(periods_per_year)
    sharpe_zero_rf = (
        portfolio_returns.mean() * periods_per_year / annualized_volatility
        if annualized_volatility > 0 else np.nan
    )

    summary_rows.append({
        "Portföy": f"İlk {portfolio_size} Hisse",
        "Dönem Sayısı": len(group),
        "Başlangıç ($)": 10_000,
        "Model Son Değer ($)": final_portfolio_value,
        "S&P 500 Son Değer ($)": final_benchmark_value,
        "Model Yıllık Getiri (%)": annualized_return * 100,
        "S&P 500 Yıllık Getiri (%)": benchmark_annualized * 100,
        "Yıllık Endeks Üstü Fark (puan)": (annualized_return - benchmark_annualized) * 100,
        "Kazançlı Dönem Oranı (%)": group["positive_period"].mean() * 100,
        "Endeksi Geçme Oranı (%)": group["beat_sp500"].mean() * 100,
        "Maksimum Düşüş (%)": calculate_max_drawdown_from_returns(portfolio_returns),
        "Yıllık Oynaklık (%)": annualized_volatility * 100,
        "Sharpe (Risksiz Faiz Hariç)": sharpe_zero_rf,
        "Ortalama Devir Oranı (%)": group["turnover_pct"].mean(),
        "Toplam Tahmini Maliyet ($)": 10_000 * (group["transaction_cost_pct"] / 100).sum()
    })

    curve = group[["holding_end_date"]].copy()
    curve["portfolio_size"] = portfolio_size
    curve["Model Portföyü ($)"] = 10_000 * (1 + portfolio_returns).cumprod().values
    curve["S&P 500 ($)"] = 10_000 * (1 + benchmark_returns).cumprod().values
    equity_curve_rows.append(curve)

backtest_summary = pd.DataFrame(summary_rows).round(2)
backtest_equity_curve = pd.concat(equity_curve_rows, ignore_index=True)

display(backtest_summary)

best_portfolio_row = backtest_summary.sort_values(
    ["Yıllık Endeks Üstü Fark (puan)", "Maksimum Düşüş (%)"],
    ascending=[False, False]
).iloc[0]
print("Geçmiş testte en yüksek endeks üstü fark:", best_portfolio_row["Portföy"])
print("Not: Bu seçim gelecekte aynı sonucu garanti etmez.")

## 34. Büyüme ve dönemsel karşılaştırma grafikleri

İlk grafik 10.000 doların model portföyü ve S&P 500'de nasıl büyüdüğünü gösterir. İkinci grafik, 30 hisselik varsayılan stratejinin her üç aylık dönemde endekse karşı farkını gösterir.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for portfolio_size, curve in backtest_equity_curve.groupby("portfolio_size"):
    axes[0].plot(
        curve["holding_end_date"],
        curve["Model Portföyü ($)"],
        label=f"Model İlk {portfolio_size}"
    )

benchmark_curve = backtest_equity_curve[
    backtest_equity_curve["portfolio_size"] == 30
]
axes[0].plot(
    benchmark_curve["holding_end_date"],
    benchmark_curve["S&P 500 ($)"],
    label="S&P 500",
    color="black",
    linewidth=2,
    linestyle="--"
)
axes[0].set_title("10.000 $ Bileşik Portföy Değeri")
axes[0].set_ylabel("Portföy Değeri ($)")
axes[0].grid(alpha=0.25)
axes[0].legend()

default_periods = portfolio_periods[portfolio_periods["portfolio_size"] == 30]
bar_colors = np.where(default_periods["excess_return_pct"] >= 0, "#16a34a", "#dc2626")
axes[1].bar(
    default_periods["holding_end_date"],
    default_periods["excess_return_pct"],
    width=55,
    color=bar_colors
)
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("İlk 30 Portföyünün Üç Aylık Endeks Üstü Getirisi")
axes[1].set_ylabel("Yüzde Puan")
axes[1].grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()

## 35. Site ve GitHub için Rev 2 dışa aktarma

Bu hücre doğrudan fiyat üreten Rev 2 regresyon modelini, sınıflandırma modelini, güncel sonuç tablosunu ve ilk tahmin kayıtlarını tek ZIP dosyasına aktarır. İlk tahmin kayıtları daha sonraki güncellemelerde ezilmemeli; aynı `forecast_id` korunarak aylık takip satırları eklenmelidir.


In [30]:
import os
import json
import shutil
import joblib
from google.colab import files

EXPORT_FOLDER = "/content/sp500_rev2_export"
ZIP_PATH = "/content/sp500_rev2_export.zip"
os.makedirs(EXPORT_FOLDER, exist_ok=True)

joblib.dump(final_regression_model_v2, f"{EXPORT_FOLDER}/regression_model_v2.joblib")
joblib.dump(final_classifier_model, f"{EXPORT_FOLDER}/classification_model.joblib")

metadata = {
    "model_version": "2.0",
    "regression_target": "target_stock_return_12m_pct",
    "numeric_features": list(numeric_features),
    "categorical_features": list(categorical_features),
    "classification_threshold": float(best_classification_threshold),
    "export_date_utc": pd.Timestamp.utcnow().isoformat(),
    "backtest_method": "quarterly_walk_forward_ml_only",
    "backtest_portfolio_sizes": BACKTEST_PORTFOLIO_SIZES,
    "backtest_holding_months": BACKTEST_HOLDING_MONTHS,
    "backtest_transaction_cost_rate": BACKTEST_TRANSACTION_COST_RATE,
    "backtest_max_stock_weight": BACKTEST_MAX_STOCK_WEIGHT,
    "backtest_max_sector_weight": BACKTEST_MAX_SECTOR_WEIGHT,
    "backtest_limitations": [
        "current_constituent_survivorship_bias",
        "no_point_in_time_analyst_or_fundamental_history"
    ]
}
with open(f"{EXPORT_FOLDER}/model_metadata_v2.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

site_columns = [
    "ticker", "yf_ticker", "company_name", "sector_sp500", "industry_sp500",
    "current_price", "market_cap", "analyst_target_mean", "analyst_target_median",
    "analyst_target_low", "analyst_target_high", "analyst_upside_pct", "analyst_count",
    "analyst_dispersion_pct", "fundamental_target_mean", "fundamental_upside_pct",
    "target_price_pe", "target_price_ev_ebitda", "target_price_fcf", "valuation_method_count",
    "forward_pe", "sector_forward_pe", "forward_eps", "fcf_yield_pct",
    "revenue_growth_pct", "earnings_growth_pct", "ebitda_margin_pct", "net_margin_pct",
    "roe_pct", "net_debt_ebitda", "return_3m_pct", "return_6m_pct", "return_12m_pct",
    "volatility_12m_pct", "max_drawdown_12m_pct", "ml_start_price",
    "ml_target_price_12m", "ml_predicted_return_12m_pct", "ml_outperform_probability_pct",
    "ml_rank_percentile", "ml_signal", "ml_forecast_date", "ml_maturity_date",
    "fundamental_score", "analyst_score", "ml_score", "quality_score", "consistency_score",
    "risk_safety_score", "data_completeness_pct", "final_score", "sp500_rank",
    "final_rank_percentile", "final_category", "qualified_strong_candidate"
]
available = [c for c in site_columns if c in sp500_snapshot.columns]
site_results = sp500_snapshot[available].copy().sort_values("sp500_rank")
for column in site_results.columns:
    if isinstance(site_results[column].dtype, pd.CategoricalDtype):
        site_results[column] = site_results[column].astype(str)
    if pd.api.types.is_datetime64_any_dtype(site_results[column]):
        site_results[column] = site_results[column].dt.strftime("%Y-%m-%d")

site_results.to_json(f"{EXPORT_FOLDER}/latest_results_v2.json", orient="records", force_ascii=False, indent=2)
site_results.to_csv(f"{EXPORT_FOLDER}/latest_results_v2.csv", index=False, encoding="utf-8-sig")

forecast_rows = []
for row in site_results.dropna(subset=["ml_start_price", "ml_target_price_12m"]).to_dict("records"):
    forecast_date = str(row.get("ml_forecast_date"))[:10]
    forecast_rows.append({
        "forecast_id": f"{row['ticker']}_{forecast_date}",
        "ticker": row["ticker"],
        "forecast_date": forecast_date,
        "maturity_date": str(row.get("ml_maturity_date"))[:10],
        "start_price": round(float(row["ml_start_price"]), 4),
        "ml_target_price_12m": round(float(row["ml_target_price_12m"]), 4),
        "predicted_return_12m_pct": round(float(row["ml_predicted_return_12m_pct"]), 2),
        "status": "ACTIVE",
        "monthly_tracking": []
    })

with open(f"{EXPORT_FOLDER}/forecasts_initial.json", "w", encoding="utf-8") as file:
    json.dump(forecast_rows, file, ensure_ascii=False, indent=2)

sp500_snapshot.to_parquet(f"{EXPORT_FOLDER}/sp500_snapshot_full_v2.parquet", index=False)
test_evaluation.to_csv(f"{EXPORT_FOLDER}/test_evaluation_v2.csv", index=False, encoding="utf-8-sig")

# Walk-forward portföy testi çıktıları
backtest_summary.to_csv(
    f"{EXPORT_FOLDER}/backtest_summary_v2.csv", index=False, encoding="utf-8-sig"
)
portfolio_periods.to_csv(
    f"{EXPORT_FOLDER}/backtest_periods_v2.csv", index=False, encoding="utf-8-sig"
)
portfolio_holdings.to_csv(
    f"{EXPORT_FOLDER}/backtest_holdings_v2.csv", index=False, encoding="utf-8-sig"
)
backtest_equity_curve.to_csv(
    f"{EXPORT_FOLDER}/backtest_equity_curve_v2.csv", index=False, encoding="utf-8-sig"
)

with open(f"{EXPORT_FOLDER}/backtest_summary_v2.json", "w", encoding="utf-8") as file:
    json.dump(
        json.loads(backtest_summary.to_json(orient="records", force_ascii=False)),
        file, ensure_ascii=False, indent=2
    )

if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)
shutil.make_archive("/content/sp500_rev2_export", "zip", "/content", "sp500_rev2_export")
print("Rev 2 dışa aktarma tamamlandı:", ZIP_PATH)
print("İlk tahmin kaydı:", len(forecast_rows))
files.download(ZIP_PATH)


Rev 2 dışa aktarma tamamlandı: /content/sp500_rev2_export.zip
İlk tahmin kaydı: 499


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>